# 第 82 课｜混沌演练、灾备容量与恢复证据

**从第 81 课的双副本持久恢复，进一步学习“有边界地注入故障，并证明恢复真的有效”。**

主线：**实验范围与假设 → 故障白名单 → 真实 KV 容量门槛 → 组合故障 → 同步/异步 RPO → 安全停止 → WAL/KV 冷重建 → 证据链与验收**。

本课完整离线：包含全部源码，不依赖第81课或其他课程的文件。使用现有 PyTorch 和标准库即可。在 CPU 运行真实因果注意力、Q/V LoRA、物理 KV 页、两个本地实例目录、WAL 与快照；两个实例仍属于**同一进程、同一个文件系统**，不是生产级分布式共识或云端故障注入。

默认生成九个彼此隔离的演练案例，并保存每个案例的完整日志、故障证据及独立重载结果。所有 RTO/RPO 数值使用**教学逻辑 tick 和事件数**，不是墙钟秒数、GPU 吞吐或服务商可用性承诺。

## 0. 运行方式与边界

从全新内核按顺序执行即可。默认运行完整九案例；如果只想先研究公式，把实验单元的 `RUN_FULL_DRILLS` 改为 `False`，保留固定检查与三道练习。你可以先打印每个案例的恢复假设，然后再读代码。

配套独立脚本也可以运行：

```bash
python lesson_82_chaos_drills_dr_capacity_evidence.py --checks-only
python lesson_82_chaos_drills_dr_capacity_evidence.py --math-only
python lesson_82_chaos_drills_dr_capacity_evidence.py
python lesson_82_chaos_drills_dr_capacity_evidence.py --reload-run lesson_82_outputs/实际运行目录
```

全程不下载模型，不联系云 API，不触碰真实主机或网络。故障注入只修改本地教学对象／临时沙盒。输出目录在当前工作路径下以 UTC 时间戳区分，避免覆盖。不要拿这份教学代码直接对线上服务注入故障。

In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""第82课：混沌演练、灾备容量与恢复证据。

完整离线、CPU PyTorch：真实自回归Transformer、Q/V LoRA与物理KV张量。
承接逻辑WAL与物理KV，引入受控故障注入、灾备容量预算、安全中止与证据审计。
仅为单进程模拟的双实例；不提供真实集群共识、远端存储、网络或IAM。
所有RTO/RPO单位是显式的教学逻辑tick；不宣称GPU毫秒性能。
"""
from __future__ import annotations

import argparse
import copy
import hashlib
import json
import math
import os
import platform
import random
import re
import tempfile
import time
from collections import Counter
from dataclasses import dataclass, asdict, field
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Callable

try:
    import torch
    from torch import nn
    from torch.nn import functional as F
except ImportError as exc:
    raise SystemExit('需要已有PyTorch环境；不会自动安装或下载。') from exc

VERSION = 'lesson82-chaos-capacity-v1'
SOURCES = {
    'aws_fis': 'https://docs.aws.amazon.com/fis/latest/userguide/what-is.html',
    'aws_fis_stop': 'https://docs.aws.amazon.com/fis/latest/userguide/stop-conditions.html',
    'azure_reliability': 'https://learn.microsoft.com/en-us/azure/well-architected/reliability/reliability-test',
    'aws_recovery': 'https://docs.aws.amazon.com/wellarchitected/2024-06-27/framework/rel-failmgmt.html',
    'torch_random': 'https://docs.pytorch.org/docs/stable/notes/randomness.html',
}

class LessonError(ValueError):
    """拒绝含糊的token边界、停止状态、报告格式或资源预算。"""


def require(ok: bool, msg: str) -> None:
    if not bool(ok):
        raise LessonError(msg)


def canonical(value: Any) -> str:
    return json.dumps(value, ensure_ascii=False, sort_keys=True,
                      separators=(",", ":"), allow_nan=False)


def digest(value: Any) -> str:
    return hashlib.sha256(canonical(value).encode("utf-8")).hexdigest()


def write_json(path: Path, value: Any) -> None:
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2, sort_keys=True,
                               allow_nan=False) + "\n", encoding="utf-8")


def bounded_str(value: Any, length: int, label: str) -> str:
    require(type(value) is str and len(value.encode("utf-8")) <= length,
            label + "必须是有界字符串")
    return value


def seed_all(seed: int) -> None:
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def state_hash(state: dict[str,torch.Tensor]) -> str:
    h=hashlib.sha256()
    for key,tensor in sorted(state.items()):
        t=tensor.detach().cpu().contiguous()
        h.update(canonical([key,str(t.dtype),list(t.shape)]).encode())
        h.update(t.reshape(-1).view(torch.uint8).numpy().tobytes())
    return h.hexdigest()

In [2]:
torch.set_num_threads(1)
seed_all(8201)
RUN_FULL_DRILLS = True
OUTPUT_DIR = Path('lesson_82_outputs')
print('Python:',platform.python_version())
print('PyTorch:',torch.__version__)
print('CUDA可用:',torch.cuda.is_available())
print('范围：隔离本地文件与CPU物理KV，不是两台真正主机。')

Python: 3.13.5
PyTorch: 2.10.0+cpu
CUDA可用: False
范围：隔离本地文件与CPU物理KV，不是两台真正主机。


## 1. 先决定演练要证明什么

好的故障演练必须预先回答：稳定状态是什么？注入什么故障？受影响范围多大？何时停止？要验证哪些恢复指标？

本课将这些问题写成严格的计划，默认仅允许 `isolated_lab_only`。故障动作白名单只包含一台主机下线、备用 WAL 断尾、备用 LoRA 权重漂移，以及备用容量变紧。它不是云平台故障注入器。

**被拒绝的演练也是有意义的结果：** 例如两个副本都下线、可信模型身份不一致、容量不足或异步同步缺口时，正确行为可能是停止自动接管，并向操作员报告风险，而非强行让系统“恢复成功”。

AWS FIS 将实验目标、动作、停止条件与安全控制明确分离，并建议先在预生产环境验证。[AWS FIS 实验安全](https://docs.aws.amazon.com/fis/latest/userguide/what-is.html) · [停止条件](https://docs.aws.amazon.com/fis/latest/userguide/stop-conditions.html)

In [3]:
class ByteMergeChatTokenizer:
    """自建教学分词器：控制token独立于用户文本，普通文本按UTF-8字节合并。

    不假装实现HF的FastTokenizer或任意模型Chat Template。固定控制ID，避免把用户
    输入里的字面控制符误当结构token。真实应用必须使用对应模型的官方tokenizer/template。
    """
    pad_token_id = 0
    bos_token_id = 1
    eos_token_id = 2
    im_start_token_id = 3
    im_end_token_id = 4
    role_ids = {"system":5, "user":6, "assistant":7}
    byte_offset = 16
    merges = (b"OK", b" 42", b"42", b"hello", b"world", b"\n", b"answer", b"JSON", b"{\"answer\":")

    def __init__(self):
        self.merge_ids = {piece: self.byte_offset + 256 + i for i,piece in enumerate(self.merges)}
        self.ids_to_merge = {v:k for k,v in self.merge_ids.items()}
        self.vocab_size = self.byte_offset + 256 + len(self.merges)
        self.specials = {0:"<pad>",1:"<bos>",2:"<eos>",3:"<im_start>",
                         4:"<im_end>",5:"<system>",6:"<user>",7:"<assistant>"}

    def encode(self, text: str, *, add_special_tokens: bool=False) -> list[int]:
        bounded_str(text, 4096, "文本")
        raw = text.encode("utf-8")
        result = []
        index = 0
        while index < len(raw):
            matches = [piece for piece in self.merges if raw.startswith(piece, index)]
            if matches:
                piece = max(matches, key=len)
                result.append(self.merge_ids[piece]);index += len(piece)
            else:
                result.append(self.byte_offset + raw[index]);index += 1
        return ([self.bos_token_id] if add_special_tokens else []) + result

    def decode(self, ids: list[int], *, skip_special_tokens: bool=False) -> str:
        require(type(ids) is list, "token IDs必须是列表")
        raw = bytearray()
        for i in ids:
            require(type(i) is int and 0 <= i < self.vocab_size, "token ID越界")
            if i in self.specials:
                if skip_special_tokens:
                    continue
                raise LessonError("正文decode不允许控制token；传入skip_special_tokens=True")
            if i in self.ids_to_merge: raw.extend(self.ids_to_merge[i])
            elif self.byte_offset <= i < self.byte_offset + 256: raw.append(i - self.byte_offset)
            else: raise LessonError("保留ID不是合法正文token")
        try:
            return bytes(raw).decode("utf-8",errors="strict")
        except UnicodeDecodeError as exc:
            raise LessonError("不完整UTF-8回答；不要用replacement字符伪装成功") from exc

    def apply_chat_template(self, messages: list[dict], *, tokenize: bool=True,
                            add_generation_prompt: bool=False):
        require(type(messages) is list and bool(messages), "messages需要非空列表")
        require(type(add_generation_prompt) is bool, "generation prompt必须是bool")
        ids = [self.bos_token_id]
        for j,m in enumerate(messages):
            require(type(m) is dict and set(m)=={"role","content"}, "message字段必须恰好role/content")
            require(m["role"] in self.role_ids and type(m["content"]) is str, "角色或内容非法")
            require(not (j == len(messages)-1 and add_generation_prompt and m["role"]=="assistant"),
                    "若最后已有assistant，请使用续写语义，不要再添加generation prompt")
            ids.extend([self.im_start_token_id,self.role_ids[m["role"]]])
            ids.extend(self.encode(m["content"],add_special_tokens=False))
            ids.append(self.eos_token_id if m["role"]=="assistant" else self.im_end_token_id)
        if add_generation_prompt:
            ids.extend([self.im_start_token_id,self.role_ids["assistant"]])
        if tokenize:
            return ids
        # 仅用于人类阅读，控制符是标记，不要将其当普通文本用encode()重新编码。
        visible = []
        for i in ids:
            if i in self.specials: visible.append(self.specials[i])
            else: visible.append(self.decode([i],skip_special_tokens=True) if
                                 self._token_is_full_utf8(i) else "<byte-part>")
        return "".join(visible)

    def _token_is_full_utf8(self, i):
        if i in self.ids_to_merge:
            try: self.ids_to_merge[i].decode("utf-8");return True
            except UnicodeError: return False
        return self.byte_offset <= i < self.byte_offset + 128

    def contract(self) -> dict:
        return {"type":"toy-byte-merge", "specials":self.specials,
                "merges":[p.hex() for p in self.merges],"vocab":self.vocab_size,
                "prefix":"BOS+message start/role/body+end+assistant header",
                "eos":self.eos_token_id,"pad":self.pad_token_id}


@dataclass(frozen=True)
class ToyConfig:
    vocab_size: int
    dim: int=32
    heads: int=4
    layers: int=2
    ff_dim: int=64
    max_length: int=160

    def validate(self):
        require(all(type(x) is int and x>0 for x in asdict(self).values()),'配置必须为正整数')
        require(self.dim%self.heads==0 and self.vocab_size>16,
                'dim/head或词表不匹配')


class CausalSelfAttention(nn.Module):
    def __init__(self,cfg: ToyConfig):
        super().__init__()
        d=cfg.dim
        self.heads=cfg.heads
        self.q_proj=nn.Linear(d,d,bias=False)
        self.k_proj=nn.Linear(d,d,bias=False)
        self.v_proj=nn.Linear(d,d,bias=False)
        self.o_proj=nn.Linear(d,d,bias=False)

    def forward(self,x:torch.Tensor,attention_mask:torch.Tensor):
        b,t,d=x.shape;h=self.heads
        def split(layer):
            return layer(x).reshape(b,t,h,d//h).transpose(1,2)
        q,k,v=split(self.q_proj),split(self.k_proj),split(self.v_proj)
        scores=(q@k.transpose(-1,-2))/math.sqrt(d//h)
        lower=torch.ones(t,t,device=x.device,dtype=torch.bool).tril()
        allowed=lower[None,None,:,:]&attention_mask[:,None,None,:]
        weights=torch.softmax(scores.masked_fill(~allowed,float('-inf')),dim=-1)
        result=(weights@v).transpose(1,2).contiguous().reshape(b,t,d)
        return self.o_proj(result)


class TransformerBlock(nn.Module):
    def __init__(self,cfg:ToyConfig):
        super().__init__()
        self.norm1=nn.LayerNorm(cfg.dim)
        self.attn=CausalSelfAttention(cfg)
        self.norm2=nn.LayerNorm(cfg.dim)
        self.ff=nn.Sequential(nn.Linear(cfg.dim,cfg.ff_dim),nn.GELU(),
                              nn.Linear(cfg.ff_dim,cfg.dim))

    def forward(self,x,mask):
        x=x+self.attn(self.norm1(x),mask)
        return x+self.ff(self.norm2(x))


class TinyChatLM(nn.Module):
    def __init__(self,cfg:ToyConfig):
        super().__init__();cfg.validate();self.cfg=cfg
        self.embed=nn.Embedding(cfg.vocab_size,cfg.dim)
        self.position=nn.Embedding(cfg.max_length,cfg.dim)
        self.blocks=nn.ModuleList([TransformerBlock(cfg) for _ in range(cfg.layers)])
        self.norm=nn.LayerNorm(cfg.dim)
        self.head=nn.Linear(cfg.dim,cfg.vocab_size,bias=False)

    def forward(self,input_ids:torch.Tensor,attention_mask:torch.Tensor):
        require(input_ids.ndim==2 and input_ids.dtype==torch.long,'输入必须为[B,T] long')
        b,t=input_ids.shape
        require(b>0 and t>=1 and t<=self.cfg.max_length,'形状或上下文超限')
        require(attention_mask.shape==input_ids.shape and attention_mask.dtype==torch.bool,
                'attention mask形状/type不符')
        require(bool(attention_mask[:,0].all()),'不接受左补齐/空提示')
        require(not bool(((~attention_mask[:,:-1])&attention_mask[:,1:]).any()),
                '只允许右补齐')
        positions=torch.arange(t,device=input_ids.device)
        x=self.embed(input_ids)+self.position(positions)[None]
        for layer in self.blocks:x=layer(x,attention_mask)
        return self.head(self.norm(x))


class LoRALinear(nn.Module):
    def __init__(self,base:nn.Linear,rank=4,alpha=8.0):
        super().__init__()
        require(type(base) is nn.Linear and type(rank) is int and rank>0,
                '手写LoRA仅支持nn.Linear与正整数rank')
        require(type(alpha) in (int,float) and math.isfinite(alpha) and alpha>0,
                'LoRA alpha非法')
        self.base=base
        self.base.requires_grad_(False)
        self.rank=rank;self.alpha=float(alpha);self.enabled=True
        self.A=nn.Parameter(base.weight.new_empty(rank,base.in_features))
        self.B=nn.Parameter(base.weight.new_zeros(base.out_features,rank))
        nn.init.kaiming_uniform_(self.A,a=math.sqrt(5))

    def forward(self,x):
        result=self.base(x)
        if not self.enabled:return result
        return result+(self.alpha/self.rank)*F.linear(F.linear(x,self.A),self.B)


def inject_lora(base:TinyChatLM,rank=4,alpha=8.0) -> TinyChatLM:
    require(not any(isinstance(m,LoRALinear) for m in base.modules()),
            '不允许重复注入LoRA')
    policy=copy.deepcopy(base)
    policy.requires_grad_(False)
    for block in policy.blocks:
        block.attn.q_proj=LoRALinear(block.attn.q_proj,rank,alpha)
        block.attn.v_proj=LoRALinear(block.attn.v_proj,rank,alpha)
    return policy


def adapters(model:TinyChatLM):
    return [(n,p) for n,p in model.named_parameters() if n.endswith(('.A','.B'))]


def adapter_snapshot(model:TinyChatLM):
    return {n:p.detach().cpu().clone() for n,p in adapters(model)}


def restore_adapter(model:TinyChatLM,values:dict):
    parameters=dict(model.named_parameters())
    require(set(values)==set(adapter_snapshot(model)),'adapter键集合不匹配')
    with torch.no_grad():
        for name,v in values.items():
            require(v.shape==parameters[name].shape and bool(torch.isfinite(v).all()),
                    'adapter参数形状/数值不匹配')
            parameters[name].copy_(v.to(parameters[name]))


def base_state(model:TinyChatLM):
    base={}
    for name,value in model.state_dict().items():
        if name.endswith(('.A','.B')):continue
        normalized=name.replace('.q_proj.base.','.q_proj.').replace('.v_proj.base.','.v_proj.')
        require(normalized not in base,'归一化基座命名冲突')
        base[normalized]=value
    return base


def base_hash(model:TinyChatLM):
    return state_hash(base_state(model))

In [4]:
tok=ByteMergeChatTokenizer()
messages=[{'role':'user','content':'hello'}]
encoded=tok.apply_chat_template(messages,tokenize=True,add_generation_prompt=True)
print('模板编码长度:',len(encoded),'前几个ID:',encoded[:12])
print('教学词表大小:',tok.vocab_size)
print('原始token ID可能泄露原文，后续精简证据不直接保存它们。')

模板编码长度: 7 前几个ID: [1, 3, 6, 275, 4, 3, 7]
教学词表大小: 281
原始token ID可能泄露原文，后续精简证据不直接保存它们。


## 2. “真实 KV”具体意味着什么？

每层真正计算 `q、k、v`，新 token 的 K/V 被写入有物理引用计数的页池，再从历史页拼接参与当前查询的注意力。模型每一层的**增量 logits 与完整上下文前向**独立比较。

页池按真实张量形状分配，检查：引用计数、空闲列表、KV 清零、事务回滚、块表合法性与终态释放。内存单位是真实 CPU tensor 字节，不使用虚构的“token=1MB”公式。它也**没有把两块真实 GPU 或两台机器合成一个集群**。

单独的容量规划还要考虑未来可生成 token：仅看当前已经占用的 KV 块，通常会低估灾备峰值。

In [5]:
@dataclass
class KVSequence:
    version: str
    tokens: list[int] = field(default_factory=list)
    block_table: list[int] = field(default_factory=list)
    pending: dict | None = None


class NoKVSpace(LessonError):
    """资源耗尽是明确失败，不能当成EOS、正常输出或偷偷丢弃KV。"""


class PagedKVPool:
    """实际CPU Tensor承载所有layer的K/V；块表是教学页索引而非vLLM kernel。

    backing: [block, layer, key_or_value, slot, kv_head, head_dim]。
    模型和版本不可在已有序列上切换；共享块只在相同版本下合法。
    尾块写入时若被共享，先Copy-on-Write复制整个块所有层再写新token。
    """
    def __init__(self, capacity_blocks: int, block_size: int, model_cfg: ToyConfig,
                 versions: set[str], dtype: torch.dtype=torch.float32):
        require(type(capacity_blocks) is int and capacity_blocks>0,'块容量必须是正整数')
        require(type(block_size) is int and block_size>0,'块token大小必须是正整数')
        model_cfg.validate()
        require(type(versions) is set and bool(versions) and all(type(v) is str and v for v in versions),
                '版本集合非法')
        require(dtype in {torch.float32,torch.float64},'教学池仅实现float32/float64')
        self.cfg=model_cfg;self.capacity_blocks=capacity_blocks;self.block_size=block_size
        self.versions=set(versions);self.dtype=dtype
        self.head_dim=model_cfg.dim//model_cfg.heads
        self.data=torch.zeros((capacity_blocks,model_cfg.layers,2,block_size,
                               model_cfg.heads,self.head_dim),dtype=dtype,device='cpu')
        self.refs=[0]*capacity_blocks
        self.free=list(range(capacity_blocks))
        self.sessions:dict[str,KVSequence]={}
        self.allocation_count=0;self.reuse_count=0;self.cow_count=0
        self.oom_count=0;self.peak_live_blocks=0;self.peak_sessions=0
        self.zeroized_releases=0

    @property
    def bytes_per_token(self):
        return 2*self.cfg.layers*self.cfg.heads*self.head_dim*self.data.element_size()

    @property
    def bytes_per_block(self):
        return self.bytes_per_token*self.block_size

    def _sid(self,sid):
        bounded_str(sid,48,'会话ID')
        require(bool(re.fullmatch(r'[A-Za-z0-9_-]+',sid)),'会话ID仅限字母数字_-')
        return sid

    def _alloc(self):
        if not self.free:
            self.oom_count+=1
            raise NoKVSpace('KV页池容量不足；拒绝本次操作且不静默覆盖缓存')
        bid=self.free.pop(0)
        require(self.refs[bid]==0,'空闲块引用数不为0')
        require(not bool(self.data[bid].count_nonzero()),'空闲块残留数据，拒绝泄漏')
        self.refs[bid]=1;self.allocation_count+=1
        self.peak_live_blocks=max(self.peak_live_blocks,self.capacity_blocks-len(self.free))
        return bid

    def _decref(self,bid):
        require(type(bid) is int and 0<=bid<self.capacity_blocks and self.refs[bid]>0,
                '引用计数下溢或块ID非法')
        self.refs[bid]-=1
        if self.refs[bid]==0:
            with torch.no_grad():self.data[bid].zero_()
            self.free.append(bid);self.free.sort();self.zeroized_releases+=1

    def open(self,sid:str,version:str):
        self._sid(sid)
        require(version in self.versions,'未知模型版本不能建缓存')
        require(sid not in self.sessions,'不能重复建同一会话')
        self.sessions[sid]=KVSequence(version=version)
        self.peak_sessions=max(self.peak_sessions,len(self.sessions))

    def fork(self,parent:str,child:str,version:str):
        self._sid(child)
        require(parent in self.sessions and child not in self.sessions,'fork父会话缺失或子会话已存在')
        p=self.sessions[parent]
        require(p.pending is None,'不能复制正在写入的缓存')
        require(version==p.version,'跨模型版本不能共享KV：即使基座相同adapter也可能不同')
        for bid in p.block_table:self.refs[bid]+=1
        self.sessions[child]=KVSequence(version=version,tokens=p.tokens.copy(),
                                        block_table=p.block_table.copy())
        self.peak_sessions=max(self.peak_sessions,len(self.sessions))
        self.audit()

    def begin(self,sid:str,token_id:int,max_length:int):
        require(sid in self.sessions,'不存在的会话')
        s=self.sessions[sid]
        require(s.pending is None,'同一会话不允许并发写入')
        require(type(token_id) is int and token_id>=0,'新token ID非法')
        require(type(max_length) is int and len(s.tokens)<max_length,'超过上下文长度')
        slot=len(s.tokens)%self.block_size
        new_page=False;cow_old=None
        # 先检查容量，错误时保留全部原有块表和引用数。
        if slot==0 or (slot!=0 and self.refs[s.block_table[-1]]>1):
            if not self.free:
                self.oom_count+=1
                raise NoKVSpace('新页或写时复制需要空闲块，但KV页池已满')
            new_page=True
        if new_page:
            bid=self._alloc()
            if slot==0:
                s.block_table.append(bid)
            else:
                cow_old=s.block_table[-1]
                with torch.no_grad():self.data[bid].copy_(self.data[cow_old])
                self._decref(cow_old)
                s.block_table[-1]=bid
                self.cow_count+=1
        else:
            bid=s.block_table[-1]
        s.pending={'token_id':token_id,'slot':slot,'block':bid,
                   'new_page':new_page,'cow_old':cow_old,'written':set()}

    def put_layer(self,sid:str,layer:int,k:torch.Tensor,v:torch.Tensor):
        require(sid in self.sessions and self.sessions[sid].pending is not None,'没有开始token写入')
        p=self.sessions[sid].pending
        require(type(layer) is int and 0<=layer<self.cfg.layers,'层ID非法')
        require(layer not in p['written'],'同一层重复写入')
        expected=(self.cfg.heads,self.head_dim)
        require(k.shape==expected and v.shape==expected and k.dtype==self.dtype and v.dtype==self.dtype,
                'K/V形状或dtype非法')
        require(k.device.type=='cpu' and v.device.type=='cpu','此教学页池只接受CPU K/V')
        require(bool(torch.isfinite(k).all()) and bool(torch.isfinite(v).all()),'K/V有非有限值')
        with torch.no_grad():
            self.data[p['block'],layer,0,p['slot']].copy_(k.detach())
            self.data[p['block'],layer,1,p['slot']].copy_(v.detach())
        p['written'].add(layer)

    def read_layer(self,sid:str,layer:int) -> tuple[torch.Tensor,torch.Tensor]:
        require(sid in self.sessions and self.sessions[sid].pending is not None,'read必须在token事务中')
        s=self.sessions[sid];p=s.pending
        require(layer in p['written'],'只能读取已写入当前token的层')
        length=len(s.tokens)+1
        ks=[];vs=[]
        for index,bid in enumerate(s.block_table):
            n=min(self.block_size,length-index*self.block_size)
            if n<=0:break
            # backing的slot轴在前，需要转成[heads,time,head_dim]。
            ks.append(self.data[bid,layer,0,:n].transpose(0,1))
            vs.append(self.data[bid,layer,1,:n].transpose(0,1))
        require(bool(ks) and sum(x.shape[1] for x in ks)==length,'KV块表与长度不同步')
        return torch.cat(ks,dim=1),torch.cat(vs,dim=1)

    def commit(self,sid):
        require(sid in self.sessions and self.sessions[sid].pending is not None,'无可提交的token')
        s=self.sessions[sid];p=s.pending
        require(p['written']==set(range(self.cfg.layers)),'有层尚未写入：禁止提交不完整KV')
        s.tokens.append(p['token_id']);s.pending=None
        self.audit()

    def abort(self,sid):
        require(sid in self.sessions and self.sessions[sid].pending is not None,'无可回滚的token')
        s=self.sessions[sid];p=s.pending
        bid=p['block'];slot=p['slot']
        if p['new_page']:
            if p['cow_old'] is not None:
                old=p['cow_old']
                s.block_table[-1]=old
                self.refs[old]+=1
            else:
                require(s.block_table[-1]==bid,'回滚末尾块不一致')
                s.block_table.pop()
            self._decref(bid)
        else:
            # 当前slot仍不属于任何已提交token，清空可能部分写入的K/V。
            with torch.no_grad():self.data[bid,:,:,slot].zero_()
        s.pending=None
        self.audit()

    def release(self,sid):
        require(sid in self.sessions,'重复释放或会话不存在')
        s=self.sessions[sid]
        require(s.pending is None,'请先回滚未完成写入再释放')
        for bid in s.block_table:self._decref(bid)
        del self.sessions[sid]
        self.audit()

    def usage(self) -> dict:
        table_ids=[b for s in self.sessions.values() for b in s.block_table]
        live=self.capacity_blocks-len(self.free)
        # 若block被两个会话引用，其有效token只计物理占用一次。
        filled={}
        for s in self.sessions.values():
            length=len(s.tokens)+(1 if s.pending else 0)
            for i,bid in enumerate(s.block_table):
                filled[bid]=max(filled.get(bid,0),min(self.block_size,length-i*self.block_size))
        occupied=sum(filled.values())
        logical_refs=len(table_ids)
        return {'capacity_blocks':self.capacity_blocks,'block_size':self.block_size,
                'free_blocks':len(self.free),'live_blocks':live,
                'physical_storage_bytes':self.data.numel()*self.data.element_size(),
                'bytes_per_token':self.bytes_per_token,'bytes_per_block':self.bytes_per_block,
                'live_reserved_bytes':live*self.bytes_per_block,
                'physical_used_slots':occupied,'live_content_bytes':occupied*self.bytes_per_token,
                'physical_internal_fragmentation_bytes':(live*self.block_size-occupied)*self.bytes_per_token,
                'logical_token_uses':sum(len(s.tokens) for s in self.sessions.values()),
                'logical_page_references':logical_refs,
                'saved_physical_blocks_by_sharing':logical_refs-live,
                'session_count':len(self.sessions),'peak_live_blocks':self.peak_live_blocks,
                'peak_sessions':self.peak_sessions,'allocation_count':self.allocation_count,
                'cow_count':self.cow_count,'oom_count':self.oom_count,
                'zeroized_releases':self.zeroized_releases}

    def audit(self) -> dict:
        expected=[0]*self.capacity_blocks
        for sid,s in self.sessions.items():
            require(s.version in self.versions,'会话绑定未知版本')
            require(len(s.block_table)==math.ceil((len(s.tokens)+(1 if s.pending else 0))/self.block_size),
                    '块表页数与有效上下文长度不一致')
            require(len(set(s.block_table))==len(s.block_table),'同一会话不能重复引用物理块')
            for bid in s.block_table:
                require(type(bid) is int and 0<=bid<self.capacity_blocks,'块表有非法物理页')
                expected[bid]+=1
        require(expected==self.refs,'物理块引用计数不等于块表的真实引用')
        require(set(self.free)=={i for i,r in enumerate(self.refs) if r==0},'free list/引用计数不一致')
        require(len(self.free)==len(set(self.free)),'空闲页重复入队')
        require(all(not bool(self.data[i].count_nonzero()) for i in self.free),
                '空闲块包含未清除的KV数据')
        u=self.usage()
        require(u['physical_internal_fragmentation_bytes']>=0,'内部碎片负数')
        require(u['live_content_bytes']<=u['live_reserved_bytes']<=u['physical_storage_bytes'],
                '实际/保留字节账目不守恒')
        return u


class CachedModelRunner:
    """一次处理一个新增token。prefill本课逐token执行，非生产高吞吐chunked prefill。

    同一层先写该token的K/V，再用块表读取全部历史K/V，对单query计算attention。
    该前缀计算和原模型full forward的掩码、位置、LoRA权重完全一致。
    """
    def __init__(self,tok:ByteMergeChatTokenizer,models:dict[str,TinyChatLM],pool:PagedKVPool):
        require(set(models)==pool.versions,'模型版本与池的版本集合不一致')
        require(all(m.cfg==pool.cfg for m in models.values()),'模型配置和池配置不匹配')
        require(pool.dtype==torch.float32,'本课模型采用float32，KV存储dtype必须一致')
        require(all(next(m.parameters()).device.type=='cpu' for m in models.values()),'教学KV池仅支持CPU')
        require(all(not p.requires_grad for m in models.values() for p in m.parameters()),
                'KV缓存仅用于冻结推理模型，不能偷偷反传训练')
        self.tok=tok;self.models=models;self.pool=pool
        self.last_logits:dict[str,torch.Tensor]={}
        self.tokens_processed=0;self.attention_query_key_products=0
        self.full_recompute_query_key_products=0
        self.model_hashes={name:state_hash(m.state_dict()) for name,m in models.items()}

    @torch.inference_mode()
    def step(self,sid:str,token_id:int) -> torch.Tensor:
        require(sid in self.pool.sessions,'cache step会话不存在')
        s=self.pool.sessions[sid]
        require(type(token_id) is int and 0<=token_id<self.pool.cfg.vocab_size,'token ID非法')
        length=len(s.tokens)
        require(length<self.pool.cfg.max_length,'上下文长度已满')
        self.pool.begin(sid,token_id,self.pool.cfg.max_length)
        model=self.models[s.version]
        try:
            ids=torch.tensor([[token_id]],dtype=torch.long)
            pos=torch.tensor([length],dtype=torch.long)
            x=model.embed(ids)+model.position(pos)[None]
            heads=self.pool.cfg.heads;head_dim=self.pool.head_dim
            for index,block in enumerate(model.blocks):
                z=block.norm1(x)
                q=block.attn.q_proj(z).reshape(heads,head_dim)
                k=block.attn.k_proj(z).reshape(heads,head_dim)
                v=block.attn.v_proj(z).reshape(heads,head_dim)
                self.pool.put_layer(sid,index,k,v)
                all_k,all_v=self.pool.read_layer(sid,index)
                # query:[heads,1,d]; keys:[heads,seq,d]；此query位于整个缓存最末尾，
                # 所以其可见范围恰好是全部已存的过去token与当前token。
                scores=(q[:,None,:]@all_k.transpose(-2,-1))/math.sqrt(head_dim)
                weights=torch.softmax(scores,dim=-1)
                value=(weights@all_v).reshape(1,1,self.pool.cfg.dim)
                x=x+block.attn.o_proj(value)
                x=x+block.ff(block.norm2(x))
                self.attention_query_key_products+=heads*(length+1)
                self.full_recompute_query_key_products+=heads*(length+1)**2
            next_logits=model.head(model.norm(x))[0,0].detach().clone()
            require(bool(torch.isfinite(next_logits).all()),'模型输出非有限logits')
            self.pool.commit(sid)
            self.last_logits[sid]=next_logits
            self.tokens_processed+=1
            return next_logits
        except Exception:
            if self.pool.sessions[sid].pending is not None:self.pool.abort(sid)
            raise

    def prefill(self,sid:str,ids:list[int]) -> torch.Tensor:
        require(type(ids) is list and bool(ids),'prefill需要非空token列表')
        require(all(type(i) is int and 0<=i<self.pool.cfg.vocab_size for i in ids),
                'prefill出现非法token')
        require(sid in self.pool.sessions,'prefill会话不存在')
        require(len(self.pool.sessions[sid].tokens)+len(ids)<=self.pool.cfg.max_length,
                'prefill不能越界或静默截断')
        last=None
        for token in ids:last=self.step(sid,token)
        return last

    def open_prefill(self,sid,version,ids):
        self.pool.open(sid,version)
        try:return self.prefill(sid,ids)
        except Exception:
            self.close(sid)
            raise

    def fork(self,parent,child,version):
        require(parent in self.last_logits,'父序列尚未prefill，不能fork')
        self.pool.fork(parent,child,version)
        self.last_logits[child]=self.last_logits[parent].clone()

    def close(self,sid):
        self.pool.release(sid)
        self.last_logits.pop(sid,None)

    @torch.inference_mode()
    def full_last(self,version:str,ids:list[int]) -> torch.Tensor:
        require(version in self.models and bool(ids),'版本或输入非法')
        model=self.models[version]
        inp=torch.tensor([ids],dtype=torch.long)
        mask=torch.ones_like(inp,dtype=torch.bool)
        return model(inp,mask)[0,-1].detach().clone()

    def assert_consistent(self,sid,atol=2e-5):
        require(sid in self.pool.sessions and sid in self.last_logits,'尚无缓存输出')
        s=self.pool.sessions[sid]
        expected=self.full_last(s.version,s.tokens)
        got=self.last_logits[sid]
        max_delta=float((expected-got).abs().max())
        require(torch.allclose(expected,got,atol=atol,rtol=2e-5),
                '真实cache输出与full recompute不一致：检查位置、KV存储或因果掩码')
        return max_delta

In [6]:
cfg=ToyConfig(tok.vocab_size)
print('隐藏维度:',cfg.dim,'层数:',cfg.layers,'注意力头:',cfg.heads)
print('上下文上限:',cfg.max_length)
print('KV每新增token的理论字节数:',cfg.layers*2*cfg.heads*(cfg.dim//cfg.heads)*4)
print('注意：这是FP32、纯KV，不包括模型权重、临时张量与CUDA allocator开销。')

隐藏维度: 32 层数: 2 注意力头: 4
上下文上限: 160
KV每新增token的理论字节数: 512
注意：这是FP32、纯KV，不包括模型权重、临时张量与CUDA allocator开销。


## 3. 模型身份：相同的 token 历史，不一定能重建相同 KV

同一请求的 token IDs 相同，不意味着随意替换 LoRA 或模型结构后仍能复用旧 K/V。恢复前要把 tokenizer、chat template、模型权重与版本一起核对。对无法通过身份验证的备份，禁止自动切主。

这里构造两套固定的随机微型模型：`stable` 与 `canary`。它们是真正会执行自回归前向的模型，但**未经预训练，也不代表实际回答质量**。教学重放默认使用确定性贪心，避免把随机采样状态遗漏伪装成一致性成功。

In [7]:
def build_servable_models(seed=7901):
    require(type(seed) is int and seed>=0,'seed非法')
    seed_all(seed)
    tok=ByteMergeChatTokenizer()
    cfg=ToyConfig(tok.vocab_size,dim=16,heads=2,layers=1,ff_dim=32,max_length=96)
    base=TinyChatLM(cfg).eval()
    stable=inject_lora(base,rank=4,alpha=8.0)
    canary=inject_lora(base,rank=4,alpha=8.0)
    gen=torch.Generator().manual_seed(seed+1)
    with torch.no_grad():
        for name,p in adapters(canary):
            if name.endswith('.B'):
                p.copy_(torch.randn(p.shape,generator=gen)*.1)
    models={'stable':stable.eval().requires_grad_(False),'canary':canary.eval().requires_grad_(False)}
    require(base_hash(models['stable'])==base_hash(models['canary']),'不是同一基座')
    require(state_hash(models['stable'].state_dict())!=state_hash(models['canary'].state_dict()),'两个版本实际权重相同')
    return tok,models

In [8]:
seed_all(8201)
tok,models=build_servable_models(8201)
identity=artifact_identity(tok,models) if 'artifact_identity' in globals() else None
print('模型版本:',list(models))
print('版本权重不同:',state_hash(models['stable'].state_dict())!=state_hash(models['canary'].state_dict()))
print('模型均冻结:',all(not p.requires_grad for m in models.values() for p in m.parameters()))

模型版本: ['stable', 'canary']
版本权重不同: True
模型均冻结: True


## 4. 为什么要有严格的 WAL 和逻辑快照？

持久的是真实请求的**逻辑状态**：模型版本、提示 token ID、生成 token ID、结束状态、事件序号和哈希前缀；K/V 张量不是快照的一部分。

恢复需要先验证完整 WAL 的顺序和 hash chain，再校验快照与可信 WAL 前缀的语义一致。通过后，依据逻辑 token 历史重新执行真实增量 KV 计算。**快照的裸 SHA-256 不等于数字签名**；如果攻击者能重写数据及摘要，还需要额外可信锚点才能防篡改。

因为 WAL 可能包含可还原用户文本的 token ID，本课只写入隔离的教学本地目录，不包含真实用户资料，也不宣称完成数据加密或访问控制。

In [9]:
ZERO_HASH='0'*64

@dataclass
class LogicalRequest:
    rid: str
    tenant: str
    key: str
    version: str
    prompt_ids: list[int]
    max_new_tokens: int
    generated: list[int]=field(default_factory=list)
    terminal: str|None=None

    def identity(self):
        return {'tenant':self.tenant,'key':self.key,'version':self.version,
                'prompt_ids':self.prompt_ids,'max_new_tokens':self.max_new_tokens}


class Ledger:
    """WAL顺序应用的逻辑状态，不包含任何模型权重或物理KV。"""
    def __init__(self,codec:ByteMergeChatTokenizer,model_identity:dict):
        self.codec=codec
        self.model_identity=copy.deepcopy(model_identity)
        self.sessions:dict[str,LogicalRequest]={}
        self.seq=0;self.last_hash=ZERO_HASH;self.tick=0

    def to_json(self):
        return {'seq':self.seq,'last_hash':self.last_hash,'tick':self.tick,
                'sessions':{key:asdict(state) for key,state in sorted(self.sessions.items())}}

    def apply(self,record:dict):
        require(type(record) is dict and set(record)=={'seq','tick','kind','payload','prev','hash'},
                'WAL字段集合错误')
        require(type(record['seq']) is int and record['seq']==self.seq+1,'WAL序号必须连续')
        require(type(record['tick']) is int and record['tick']==self.tick+1,
                '本课tick是从1开始的每条持久事件序号')
        require(record['prev']==self.last_hash,'WAL前驱hash错误或从错误分叉接入')
        require(record['hash']==digest({k:v for k,v in record.items() if k!='hash'}),
                'WAL内容hash错误')
        kind=record['kind'];p=record['payload']
        require(type(p) is dict,'payload必须是dict')
        if kind=='create':
            require(set(p)=={'rid','tenant','key','version','prompt_ids','max_new_tokens'},
                    'create字段集合错误')
            rid=make_rid(p['tenant'],p['key'])
            require(rid==p['rid'] and rid not in self.sessions,'重复idempotency key/create冲突')
            require(p['version'] in self.model_identity['weights'],'WAL引用未知权重版本')
            require(type(p['prompt_ids']) is list and len(p['prompt_ids'])>0 and
                    len(p['prompt_ids'])<self.model_identity['max_length'], '提示token非法/超上下文')
            require(all(type(t) is int and 0<=t<self.codec.vocab_size for t in p['prompt_ids']),
                    'WAL提示token超词表')
            require(type(p['max_new_tokens']) is int and 1<=p['max_new_tokens']<=4 and
                    len(p['prompt_ids'])+p['max_new_tokens']<=self.model_identity['max_length'],
                    'WAL完成长度非法')
            self.sessions[rid]=LogicalRequest(**p)
        elif kind=='token':
            require(set(p)=={'rid','index','token','terminal'},'token字段集合错误')
            rid=p['rid'];require(rid in self.sessions,'追加token的请求不存在')
            session=self.sessions[rid]
            require(session.terminal is None,'终态后不可再追加token')
            require(type(p['index']) is int and p['index']==len(session.generated),
                    'token序号不连续')
            tok=p['token'];require(type(tok) is int and 0<=tok<self.codec.vocab_size,
                                  'token ID越界')
            require(p['terminal'] in {None,'eos','length'},'非法停止原因')
            next_length=len(session.generated)+1
            expected='eos' if tok==self.codec.eos_token_id else (
                     'length' if next_length==session.max_new_tokens else None)
            require(p['terminal']==expected,'EOS、长度终止不一致')
            session.generated.append(tok)
            session.terminal=p['terminal']
        else:
            raise LessonError('WAL事件类型非法')
        self.seq=record['seq'];self.tick=record['tick'];self.last_hash=record['hash']

    @classmethod
    def from_json(cls,codec,model_identity,data):
        require(type(data) is dict and set(data)=={'seq','last_hash','tick','sessions'},
                '快照状态字段错误')
        out=cls(codec,model_identity)
        require(type(data['seq']) is int and data['seq']>=0 and
                data['tick']==data['seq'] and type(data['last_hash']) is str and
                len(data['last_hash'])==64,'快照head错误')
        out.seq=data['seq'];out.tick=data['tick'];out.last_hash=data['last_hash']
        require(type(data['sessions']) is dict,'快照会话表必须为dict')
        for rid,s in data['sessions'].items():
            require(type(s) is dict and set(s)==set(LogicalRequest.__dataclass_fields__),
                    '快照会话字段非法')
            item=LogicalRequest(**s)
            require(item.rid==rid==make_rid(item.tenant,item.key),'快照id不一致')
            require(item.version in model_identity['weights'],'快照版本已失效')
            require(item.terminal in {None,'eos','length'},'快照停止状态非法')
            require(all(type(i) is int and 0<=i<codec.vocab_size for i in
                        item.prompt_ids+item.generated),'快照token越界')
            require(1<=item.max_new_tokens<=4 and
                    len(item.generated)<=item.max_new_tokens,'快照生成长度非法')
            if item.terminal=='eos':
                require(bool(item.generated) and item.generated[-1]==codec.eos_token_id,
                        '快照EOS状态不一致')
            if item.terminal=='length':
                require(len(item.generated)==item.max_new_tokens and
                        item.generated[-1]!=codec.eos_token_id,'快照长度终止不一致')
            if item.terminal is None:
                require(codec.eos_token_id not in item.generated and
                        len(item.generated)<item.max_new_tokens,'活动快照包含终态回答')
            out.sessions[rid]=item
        return out


def make_rid(tenant:str,key:str) -> str:
    for value,label in ((tenant,'租户'),(key,'幂等键')):
        require(type(value) is str and re.fullmatch('[A-Za-z0-9_-]{1,18}',value) is not None,
                label+'只能含1..18个ASCII字母数字_-')
    return tenant+'__'+key


def atomic_json(path:Path,payload:dict) -> None:
    """单机持久文件替换＋fsync；没有提供跨机器线性一致性。"""
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_name(path.name+'.tmp')
    require(not tmp.exists(),'存在未清理的临时文件，拒绝覆盖')
    try:
        with tmp.open('x',encoding='utf-8') as f:
            f.write(json.dumps(payload,ensure_ascii=False,sort_keys=True,allow_nan=False)+'\n')
            f.flush();os.fsync(f.fileno())
        os.replace(tmp,path)
        if hasattr(os,'O_DIRECTORY'):
            fd=os.open(str(path.parent),os.O_RDONLY|os.O_DIRECTORY)
            try:os.fsync(fd)
            finally:os.close(fd)
    finally:
        if tmp.exists():tmp.unlink()


def make_record(ledger:Ledger,kind:str,payload:dict) -> dict:
    row={'seq':ledger.seq+1,'tick':ledger.tick+1,'kind':kind,
         'payload':copy.deepcopy(payload),'prev':ledger.last_hash}
    row['hash']=digest(row)
    trial=copy.deepcopy(ledger)
    trial.apply(row)
    return row


class DurableJournal:
    """严格顺序WAL：每行必须换行、hash chain，追加时flush+fsync。

    这是本地模拟“两个独立实例的磁盘”，不是分布式事务，不跨网复制fsync保证。
    故障后的未完成行默认拒绝；不静默截断或忽略。
    """
    def __init__(self,folder:Path,codec,model_identity,load_existing=False):
        self.folder=Path(folder);self.folder.mkdir(parents=True,exist_ok=True)
        self.path=self.folder/'events.jsonl'
        self.snapshot_path=self.folder/'snapshot.json'
        self.codec=codec;self.model_identity=copy.deepcopy(model_identity)
        if not load_existing:
            require(not self.path.exists() and not self.snapshot_path.exists(),
                    '实例目录已有数据，禁止当作新副本覆盖')
            self.path.write_bytes(b'')
        else:
            require(self.path.is_file(),'恢复时WAL不存在')
        self.records=[];self.ledger=Ledger(codec,model_identity)
        if load_existing:self.recover()

    def read_records(self) -> list[dict]:
        raw=self.path.read_bytes()
        require(not raw or raw.endswith(b'\n'),'WAL末尾存在截断行，必须人工修复/显式恢复流程')
        rows=[];temp=Ledger(self.codec,self.model_identity)
        for n,line in enumerate(raw.splitlines(),1):
            require(line!=b'','WAL存在空行')
            try:
                row=json.loads(line.decode('utf-8'))
            except (UnicodeDecodeError,json.JSONDecodeError) as exc:
                raise LessonError(f'WAL第{n}行不是有效JSON/UTF8') from exc
            temp.apply(row)
            rows.append(row)
        return rows

    def recover(self) -> dict:
        rows=self.read_records()  # 对全WAL做语法/hash链验证，快照不绕过完整性审计。
        all_state=Ledger(self.codec,self.model_identity)
        for row in rows:all_state.apply(row)
        snapshot_used=False;tail=rows
        if self.snapshot_path.exists():
            try:wrapped=json.loads(self.snapshot_path.read_text(encoding='utf-8'))
            except (UnicodeDecodeError,json.JSONDecodeError) as exc:
                raise LessonError('快照文件无法解码') from exc
            require(type(wrapped) is dict and set(wrapped)=={'payload','sha256'},'快照封装非法')
            data=wrapped['payload']
            require(wrapped['sha256']==digest(data),'快照内容摘要不匹配')
            require(type(data) is dict and set(data)=={'version','model_identity','state'},
                    '快照字段错误')
            require(data['version']==VERSION and data['model_identity']==self.model_identity,
                    '快照模型/分词身份不匹配')
            restored=Ledger.from_json(self.codec,self.model_identity,data['state'])
            require(restored.seq<=len(rows),'快照seq超出WAL实际长度')
            prefix=Ledger(self.codec,self.model_identity)
            for row in rows[:restored.seq]:prefix.apply(row)
            require(prefix.to_json()==restored.to_json(),
                    '快照与WAL前缀的内容不一致：不允许信任重算hash后的假快照')
            tail=rows[restored.seq:]
            for row in tail:restored.apply(row)
            require(restored.to_json()==all_state.to_json(),'快照加尾WAL不等于全量恢复')
            self.ledger=restored;snapshot_used=True
        else:self.ledger=all_state
        self.records=rows
        return {'wal_records_verified':len(rows),'snapshot_used':snapshot_used,
                'tail_events_semantically_applied':len(tail),'head_hash':self.ledger.last_hash}

    def append_row(self,row:dict) -> None:
        # 在触盘前先检查下一状态，避免错误事件落盘后再被拒绝。
        probe=copy.deepcopy(self.ledger);probe.apply(row)
        with self.path.open('ab') as f:
            payload=(canonical(row)+'\n').encode('utf-8')
            f.write(payload);f.flush();os.fsync(f.fileno())
        self.ledger=probe
        self.records.append(copy.deepcopy(row))

    def checkpoint(self) -> dict:
        data={'version':VERSION,'model_identity':copy.deepcopy(self.model_identity),
              'state':self.ledger.to_json()}
        atomic_json(self.snapshot_path,{'payload':data,'sha256':digest(data)})
        return {'seq':self.ledger.seq,'last_hash':self.ledger.last_hash,
                'size_bytes':self.snapshot_path.stat().st_size,
                'contents':'logical token IDs and request states; NOT raw KV tensors'}

## 5. 独立副本与单主控制者

每个副本拥有独立 WAL 文件、KV 页池和模型实例。同步模式下，一次 ACK 要等两侧持久化完成；异步模式允许备份滞后，因此接管时可能必须明确声明已确认事件损失。

单个可信教学控制指针维护 epoch 并拒绝旧主写入。它不是多个节点之间的共识协议，不能替代网络分区下的真实外部 fencing、数据库 CAS 或租约服务。

若写盘成功但 ACK 未送达，重试必须使用同一个幂等键对账，不能无条件重新执行生成副作用。

In [10]:
def artifact_identity(tok,models:dict[str,TinyChatLM]) -> dict:
    require(type(models) is dict and set(models)=={'stable','canary'},'模型版本集合不完整')
    configs=[asdict(m.cfg) for m in models.values()]
    require(configs[0]==configs[1],'两个版本配置不同')
    return {'tokenizer':digest(tok.contract()),'weights':{k:state_hash(m.state_dict())
              for k,m in sorted(models.items())},'config':configs[0],
              'max_length':configs[0]['max_length']}


class Replica:
    def __init__(self,name:str,folder:Path,tok,models,capacity_blocks=64,
                 block_size=4,load_existing=False):
        require(name in {'A','B'},'副本名必须A/B')
        self.name=name;self.tok=tok;self.models=copy.deepcopy(models)
        self.identity=artifact_identity(tok,self.models)
        self.folder=Path(folder)
        self.journal=DurableJournal(self.folder,tok,self.identity,load_existing)
        cfg=next(iter(self.models.values())).cfg
        self.pool=PagedKVPool(capacity_blocks,block_size,cfg,set(self.models))
        self.runner=CachedModelRunner(tok,self.models,self.pool)
        self.up=True
        self.last_recovery={'wal_records_verified':0,'snapshot_used':False,
                            'tail_events_semantically_applied':0}
        if load_existing:self.recover_cache()

    @property
    def ledger(self):return self.journal.ledger

    def _expected(self,rid):
        require(rid in self.pool.sessions and rid in self.runner.last_logits,
                '活动请求没有对应真实KV；拒绝偷偷从空上下文解码')
        logits=self.runner.last_logits[rid]
        require(bool(torch.isfinite(logits).all()),'KV输出异常')
        return int(logits.argmax())

    def prevalidate(self,row:dict) -> None:
        require(self.up,'实例已宕机')
        require(artifact_identity(self.tok,self.models)==self.identity,
                '运行模型权重漂移，拒绝生成/应用新事件')
        test=copy.deepcopy(self.ledger);test.apply(row)
        kind=row['kind'];payload=row['payload']
        if kind=='token':
            rid=payload['rid']
            require(rid in self.pool.sessions and rid in self.runner.last_logits,
                    '副本缺失活动KV：先恢复再继续')
            require(self._expected(rid)==payload['token'],
                    '复制到另一个副本时增量输出不一致：权重/前缀/位置存在漂移')

    def accept(self,row:dict) -> None:
        self.prevalidate(row)
        self.journal.append_row(row)
        p=row['payload']
        try:
            if row['kind']=='create':
                self.runner.open_prefill(p['rid'],p['version'],p['prompt_ids'])
            else:
                if p['terminal'] is None:self.runner.step(p['rid'],p['token'])
                else:self.runner.close(p['rid'])
            self.pool.audit()
        except Exception:
            # 已持久事件不能撒谎称“未发生”；该副本必须标记不可用并离线重建。
            self.up=False
            raise

    def create(self,payload:dict) -> dict:
        record=make_record(self.ledger,'create',payload)
        self.accept(record)
        return record

    def token(self,payload:dict) -> dict:
        record=make_record(self.ledger,'token',payload)
        self.accept(record)
        return record

    def recover_cache(self) -> dict:
        require(artifact_identity(self.tok,self.models)==self.identity,
                '恢复实例权重身份不匹配')
        details=self.journal.recover()
        # 丢弃所有易失KV，依据逻辑会话从最早提示重建。
        cfg=next(iter(self.models.values())).cfg
        self.pool=PagedKVPool(self.pool.capacity_blocks,self.pool.block_size,cfg,set(self.models))
        self.runner=CachedModelRunner(self.tok,self.models,self.pool)
        total=0;active=0
        for rid,state in sorted(self.ledger.sessions.items()):
            if state.terminal is not None:continue
            active+=1
            self.runner.open_prefill(rid,state.version,state.prompt_ids)
            total+=len(state.prompt_ids)
            for t in state.generated:
                require(self._expected(rid)==t,
                        '恢复后同一前缀的贪心输出不同，拒绝身份/模型漂移')
                self.runner.step(rid,t)
                total+=1
            self.runner.assert_consistent(rid)
        self.up=True
        details.update({'active_sessions_rebuilt':active,'tokens_recomputed':total,
                        'kv_tensors_restored_from_disk':False,'kv_live_blocks':self.pool.usage()['live_blocks']})
        self.last_recovery=details
        return details

    def audit(self):
        active={k for k,v in self.ledger.sessions.items() if v.terminal is None}
        require(set(self.pool.sessions)==active and set(self.runner.last_logits)==active,
                '逻辑活跃会话与KV实体不一致')
        max_diff=0.0
        for rid in sorted(active):
            st=self.ledger.sessions[rid]
            require(self.pool.sessions[rid].tokens==st.prompt_ids+st.generated,
                    'KV与已确认的逻辑token历史不一致')
            require(self.pool.sessions[rid].version==st.version,'KV与持久版本绑定不一致')
            max_diff=max(max_diff,self.runner.assert_consistent(rid))
        u=self.pool.audit()
        return {'replica':self.name,'logical_seq':self.ledger.seq,'logical_requests':len(self.ledger.sessions),
                'active_sessions':len(active),'kv_live_blocks':u['live_blocks'],
                'max_cache_full_logits_diff':max_diff,'models':self.identity['weights']}


def _request_payload(tok,tenant,key,version,text,max_new_tokens):
    rid=make_rid(tenant,key)
    require(version in {'stable','canary'},'模型版本非法')
    require(type(max_new_tokens) is int and 1<=max_new_tokens<=4,'输出预算非法')
    prompt=tok.apply_chat_template([{'role':'user','content':text}],tokenize=True,
                                   add_generation_prompt=True)
    return {'rid':rid,'tenant':tenant,'key':key,'version':version,
            'prompt_ids':prompt,'max_new_tokens':max_new_tokens}

In [11]:
class SimulatedCrash(RuntimeError):
    """明确模拟写盘之后、客户端收到ACK之前的中断，不执行真实进程杀死。"""


def save_model_artifacts(path:Path,tok,models) -> dict:
    require(not Path(path).exists(),'模型artifact不可覆盖')
    identity=artifact_identity(tok,models)
    blob={'version':VERSION,'identity':identity,'tokenizer':tok.contract(),
          'weights':{version:{k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
                     for version,model in models.items()}}
    torch.save(blob,path)
    return identity


def load_model_artifacts(path:Path):
    # 仅加载本课自己生成、可信来源的纯张量与dict；不执行外部网络下载。
    blob=torch.load(path,map_location='cpu',weights_only=True)
    require(blob['version']==VERSION,'artifact版本不匹配')
    tok=ByteMergeChatTokenizer()
    require(blob['tokenizer']==tok.contract(),'tokenizer/chat template身份不匹配')
    identity=blob['identity']
    cfg=ToyConfig(**identity['config'])
    models={}
    for name,values in blob['weights'].items():
        model=inject_lora(TinyChatLM(cfg),rank=4,alpha=8.0)
        model.load_state_dict(values,strict=True)
        models[name]=model.eval().requires_grad_(False)
    require(artifact_identity(tok,models)==identity,'模型artifact哈希不匹配')
    return tok,models


class Coordinator:
    """一个教学用串行控制器，持久化epoch与ACK审计；不是分布式共识。"""
    def __init__(self,root:Path,tok,models,policy='sync',seed=8101,
                 capacity_blocks=64,load_existing=False):
        require(policy in {'sync','async'},'确认模式只能sync/async')
        self.root=Path(root);self.root.mkdir(parents=True,exist_ok=True)
        self.control_path=self.root/'control.json'
        self.artifact_path=self.root/'models.pt'
        self.tok=tok;self.models=models
        self.model_identity=artifact_identity(tok,models)
        self.capacity_blocks=capacity_blocks
        self.seed=seed
        if load_existing:
            self.control=self._read_control()
            require(self.control['model_identity']==self.model_identity,
                    '重载模型与原先已确认事件的权重/词表身份不符')
            require(self.control['policy']==policy,'控制器确认策略不可暗中更改')
        else:
            require(not self.control_path.exists() and not self.artifact_path.exists(),
                    '根目录已包含集群持久状态，禁止覆盖')
            save_model_artifacts(self.artifact_path,tok,models)
            self.control={'version':VERSION,'active':'A','epoch':1,'policy':policy,
                          'model_identity':self.model_identity,'ack_seq':0,'ack_tick':0,
                          'ack_history':[],'failovers':[],'quarantined':[],'seed':seed,
                          'capacity_blocks':capacity_blocks}
            atomic_json(self.control_path,self.control)
        self.rep={name:Replica(name,self.root/name,tok,models,capacity_blocks,
                               load_existing=load_existing) for name in ('A','B')}
        self.updates_acknowledged=0
        for name in self.control['quarantined']:
            require(name!=self.control['active'],'活跃主不能被隔离')
            self.rep[name].shutdown()
        self._check_artifacts()

    def _read_control(self):
        try:state=json.loads(self.control_path.read_text(encoding='utf-8'))
        except (UnicodeDecodeError,json.JSONDecodeError,FileNotFoundError) as exc:
            raise LessonError('控制指针无法读取') from exc
        require(type(state) is dict and
                set(state)=={'version','active','epoch','policy','model_identity','ack_seq',
                             'ack_tick','ack_history','failovers','quarantined','seed','capacity_blocks'},
                '控制记录字段非法')
        require(state['version']==VERSION and state['active'] in {'A','B'} and
                type(state['epoch']) is int and state['epoch']>=1,'控制代际非法')
        require(type(state['ack_seq']) is int and state['ack_seq']>=0 and
                state['ack_tick']==state['ack_seq'],'ACK水位非法')
        require(type(state['quarantined']) is list and
                set(state['quarantined'])<={'A','B'} and
                state['active'] not in state['quarantined'],
                '隔离副本列表非法')
        return state

    def _check_artifacts(self):
        token,models=load_model_artifacts(self.artifact_path)
        require(artifact_identity(token,models)==self.model_identity,
                '模型artifact与当前控制身份不符')
        for r in self.rep.values():
            require(r.identity==self.model_identity,'副本模型身份漂移')
            require(artifact_identity(r.tok,r.models)==r.identity,'运行中模型权重被改写')

    def _fence(self,epoch:int):
        durable=self._read_control()
        require(type(epoch) is int and epoch==durable['epoch'] and
                durable['active']==self.control['active'] and
                durable['epoch']==self.control['epoch'],
                'epoch过期或控制指针已被其他控制器切换；拒绝陈旧主节点写入')
        require(self.rep[durable['active']].up,'活跃副本不可用')
        return self.rep[durable['active']]

    def _standby(self):return self.rep['B' if self.control['active']=='A' else 'A']

    def replicate_pending(self):
        primary=self.rep[self.control['active']];target=self._standby()
        require(target.name not in self.control['quarantined'],
                '备机日志已隔离，必须显式丢弃旧分叉并重建')
        require(target.up,'备机不可用，不能确认同步复制')
        require(target.identity==primary.identity,'复制副本模型身份不相等')
        left=target.journal.records;right=primary.journal.records
        require(len(left)<=len(right) and
                all(a==b for a,b in zip(left,right)),
                '日志发生分叉：不可自动拼接，必须隔离并人工处理')
        count=0
        for row in right[len(left):]:
            target.accept(row);count+=1
        require(target.ledger.last_hash==primary.journal.records[target.ledger.seq-1]['hash']
                if target.ledger.seq else target.ledger.last_hash==ZERO_HASH,
                '复制完成后head hash不一致')
        return count

    def _ack(self,row:dict):
        require(type(row) is dict and 'seq' in row and 'hash' in row,'ACK没有WAL证据')
        primary=self.rep[self.control['active']]
        require(primary.ledger.seq>=row['seq'] and
                primary.journal.records[row['seq']-1]==row,'主机日志和ACK不匹配')
        already_acked=row['seq']<=self.control['ack_seq']
        if already_acked:
            require(any(e['seq']==row['seq'] and e['hash']==row['hash']
                        for e in self.control['ack_history']),
                    '历史ACK证据与当前日志不匹配')
        elif self.control['policy']=='sync':
            standby=self._standby()
            require(standby.name not in self.control['quarantined'] and
                    standby.up and standby.ledger.seq>=row['seq'] and
                    standby.journal.records[row['seq']-1]==row,'同步ACK前备机没有持久化该事件')
        if row['seq']>self.control['ack_seq']:
            require(row['seq']==self.control['ack_seq']+1,'不能越过未确认的先前事件')
            self.control['ack_seq']=row['seq'];self.control['ack_tick']=row['tick']
            self.control['ack_history'].append({'seq':row['seq'],'hash':row['hash'],
                                               'epoch':self.control['epoch']})
            atomic_json(self.control_path,self.control)
            self.updates_acknowledged+=1
        return {'seq':row['seq'],'hash':row['hash'],'epoch':self.control['epoch'],
                'durability':'primary+standby' if self.control['policy']=='sync' else 'primary_only'}

    def _commit(self,row:dict,crash_after_durable=False):
        primary=self.rep[self.control['active']]
        primary.accept(row)   # 先写盘fsync，再响应或复制。
        if self.control['policy']=='sync':self.replicate_pending()
        if crash_after_durable:
            raise SimulatedCrash('WAL已fsync但控制ACK未写入；客户端必须同键对账重试')
        return self._ack(row)

    def start(self,epoch,tenant,key,version='stable',text='hello',max_new_tokens=4,
              crash_after_durable=False):
        active=self._fence(epoch)
        payload=_request_payload(self.tok,tenant,key,version,text,max_new_tokens)
        rid=payload['rid']
        if rid in active.ledger.sessions:
            item=active.ledger.sessions[rid]
            require(item.identity()=={k:payload[k] for k in item.identity()},
                    '同一幂等键与先前请求内容/版本不一致')
            row=next(r for r in active.journal.records if r['kind']=='create'
                     and r['payload']['rid']==rid)
            if self.control['policy']=='sync' and row['seq']>self.control['ack_seq']:
                self.replicate_pending()
            ack=self._ack(row)
            return {'rid':rid,'new_event':False,'ack':ack}
        row=make_record(active.ledger,'create',payload)
        ack=self._commit(row,crash_after_durable)
        return {'rid':rid,'new_event':True,'ack':ack}

    def step(self,epoch,rid,expected_index:int,crash_after_durable=False):
        active=self._fence(epoch)
        require(rid in active.ledger.sessions,'请求不存在')
        item=active.ledger.sessions[rid]
        require(type(expected_index) is int and expected_index>=0,'续传序号非法')
        if expected_index<len(item.generated):
            row=next(r for r in active.journal.records if r['kind']=='token' and
                     r['payload']['rid']==rid and r['payload']['index']==expected_index)
            if self.control['policy']=='sync' and row['seq']>self.control['ack_seq']:
                self.replicate_pending()
            ack=self._ack(row)
            return {'rid':rid,'index':expected_index,'token':row['payload']['token'],
                    'terminal':row['payload']['terminal'],'new_event':False,'ack':ack}
        require(expected_index==len(item.generated),'不能跳过尚未产生的token')
        require(item.terminal is None,'请求已经终止')
        tok=active._expected(rid)
        stop='eos' if tok==self.tok.eos_token_id else (
             'length' if expected_index+1==item.max_new_tokens else None)
        row=make_record(active.ledger,'token',{'rid':rid,'index':expected_index,
                                              'token':tok,'terminal':stop})
        ack=self._commit(row,crash_after_durable)
        return {'rid':rid,'index':expected_index,'token':tok,'terminal':stop,
                'new_event':True,'ack':ack}

    def checkpoint(self):
        return {name:self.rep[name].journal.checkpoint() for name in ('A','B')}

    def promote(self,epoch,target='B',allow_data_loss=False,rpo_budget_ticks=0,
                rto_budget_ticks=12,tokens_per_tick=32,wal_records_per_tick=16):
        require(type(allow_data_loss) is bool,'数据丢失授权必须显式bool')
        require(target in self.rep and target!=self.control['active'],'接管目标非法')
        persisted=self._read_control()
        require(type(epoch) is int and epoch==self.control['epoch'] and
                persisted==self.control, '接管用epoch过期或控制状态发生变化')
        # 故障后原主机可down；只有这个教学控制指针继续提供围栏，不由原主授权。
        require(all(type(v) is int and v>=0 for v in (rpo_budget_ticks,rto_budget_ticks))
                and tokens_per_tick>0 and wal_records_per_tick>0,'恢复目标/预算非法')
        old_active=self.control['active'];standby=self.rep[target]
        require(standby.up,'备机不可用')
        self._check_artifacts()
        # 在新对象中只从本地持久日志及快照恢复，易失KV不复制过来。
        fresh=Replica(target,self.root/target,self.tok,self.models,self.capacity_blocks,
                      load_existing=True)
        require(artifact_identity(fresh.tok,fresh.models)==self.model_identity,
                '灾难恢复模型/模板指纹不一致')
        audit=fresh.audit()
        ack_seq=self.control['ack_seq']
        lost=max(0,ack_seq-fresh.ledger.seq)
        require(fresh.ledger.seq<=self.rep[old_active].ledger.seq,
                '备机不能凭空比主机多出旧epoch事件')
        # 完整一致的确认前缀不能只是seq相等；还要比较哈希链前缀。
        primary_records=self.rep[old_active].journal.records
        require(fresh.journal.records==primary_records[:len(fresh.journal.records)],
                '备机历史并非旧主已提交前缀；禁止接管分叉日志')
        rpo_gap=max(0,self.control['ack_tick']-fresh.ledger.tick)
        rto=2+math.ceil(fresh.last_recovery['wal_records_verified']/wal_records_per_tick)+(
             math.ceil(fresh.last_recovery['tokens_recomputed']/tokens_per_tick))
        if (lost>0 or rpo_gap>rpo_budget_ticks) and not allow_data_loss:
            raise LessonError('未复制的已ACK事件存在，自动接管必须阻断并报告RPO风险')
        if rto>rto_budget_ticks:
            raise LessonError('预计恢复逻辑tick超出RTO门槛；本课拒绝自动接管')
        report={'from':old_active,'to':target,'old_epoch':epoch,'new_epoch':epoch+1,
                'sync_mode':self.control['policy'],'ack_seq_before':ack_seq,
                'target_seq':fresh.ledger.seq,'lost_acked_events':lost,
                'rpo_gap_logical_ticks':rpo_gap,'rpo_objective_ticks':rpo_budget_ticks,
                'rto_simulated_ticks':rto,'rto_objective_ticks':rto_budget_ticks,
                'rto_formula':'2 + ceil(WAL_verified / records_per_tick) + ceil(KV_replay_tokens / tokens_per_tick)',
                'recovered_active_sessions':audit['active_sessions'],
                'recovered_tokens_recomputed':fresh.last_recovery['tokens_recomputed'],
                'data_loss_override':allow_data_loss and lost>0,
                'no_real_time_measurement':True}
        new_control=copy.deepcopy(self.control)
        new_control['active']=target;new_control['epoch']=epoch+1
        new_control['ack_seq']=fresh.ledger.seq;new_control['ack_tick']=fresh.ledger.tick
        if lost>0:
            new_control['ack_history']=[row for row in new_control['ack_history']
                                        if row['seq']<=fresh.ledger.seq]
            new_control['quarantined']=sorted(set(new_control['quarantined'])|{old_active})
        new_control['failovers'].append(copy.deepcopy(report))
        # 本地控制指针fsync后再关闭旧主；失效旧epoch依赖这个唯一可信控制点。
        atomic_json(self.control_path,new_control)
        self.rep[old_active].shutdown()
        standby.shutdown()
        self.rep[target]=fresh
        self.control=new_control
        return report

    def audit(self):
        require(self._read_control()==self.control,'控制内存与持久状态不一致')
        self._check_artifacts()
        active=self.rep[self.control['active']]
        require(active.up,'当前活跃副本不可用')
        return {'active':self.control['active'],'epoch':self.control['epoch'],
                'ack_seq':self.control['ack_seq'], 'artifacts':self.model_identity,
                'active_audit':active.audit(),
                'standby_head':self._standby().ledger.seq}


def _replica_shutdown(self):
    for sid in list(self.pool.sessions):
        if self.pool.sessions[sid].pending is not None:self.pool.abort(sid)
        self.runner.close(sid)
    require(self.pool.audit()['live_blocks']==0,'关闭副本仍有物理KV泄漏')
    self.up=False

Replica.shutdown=_replica_shutdown


def restore_controller(root:Path) -> Coordinator:
    root=Path(root)
    tok,models=load_model_artifacts(root/'models.pt')
    control=json.loads((root/'control.json').read_text(encoding='utf-8'))
    result=Coordinator(root,tok,models,policy=control['policy'],seed=control['seed'],
                       capacity_blocks=control['capacity_blocks'],load_existing=True)
    result.audit()
    return result

In [12]:
identity=artifact_identity(tok,models)
print('模型身份摘要：',identity['weights'])
print('只有共享此身份的副本才有资格接管，且必须通过WAL、容量和RPO/RTO门槛。')

模型身份摘要： {'canary': '6d0092704196ed697faa2d49de751c8bfc052b873a6fb9d2c43c4806b39cb285', 'stable': '5c6959ba7a2164fc7b555c4c24033a578f8cfabe96a080c1a7c57a56579f12c7'}
只有共享此身份的副本才有资格接管，且必须通过WAL、容量和RPO/RTO门槛。


## 6. 演练安全：范围、白名单、全局停止与证据

这一节同时实现三个独立概念：

**Safety Plan**：只允许隔离范围，有最大故障数、最大逻辑时长、会话数、KV 预留与可接受损失授权。

**Chaos Guard**：任何新故障注入前核对动作和目标，超预算／安全开关触发后拒绝后续注入。

**Evidence Trail**：记录阶段、逻辑 tick、编号、前一条哈希、当前哈希，并在每行之后 `flush+fsync`。报告只保留紧凑计数，不保存提示正文或 token ID。证据独立重载时重新扫描 hash chain。

完整控制与证据不是形式主义：AWS 和 Azure 的可靠性测试指导都强调预先定义停止条件、目标、受控范围和恢复验收。[AWS 安全停止](https://docs.aws.amazon.com/fis/latest/userguide/safety-lever.html) · [Azure 可靠性测试](https://learn.microsoft.com/en-us/azure/well-architected/reliability/reliability-test)

**注意**：当前 hash chain 的末尾锚点仍保存在同一份可变本地目录中。它能发现意外篡改和日志错配，但没有数字签名、独立审计方、远端 WORM 或不可抵赖保证。

In [13]:
def _advance(c:Coordinator,rid:str,n:int):
    saved=[]
    for _ in range(n):
        st=c.rep[c.control['active']].ledger.sessions[rid]
        if st.terminal is not None:break
        saved.append(c.step(c.control['epoch'],rid,len(st.generated)))
    return saved



def _atomic_bytes(path:Path,raw:bytes) -> None:
    path=Path(path)
    tmp=path.with_name(path.name+'.repair_tmp')
    require(not tmp.exists(),'修复临时文件未清理')
    try:
        with tmp.open('xb') as f:
            f.write(raw);f.flush();os.fsync(f.fileno())
        os.replace(tmp,path)
        if hasattr(os,'O_DIRECTORY'):
            fd=os.open(str(path.parent),os.O_RDONLY|os.O_DIRECTORY)
            try:os.fsync(fd)
            finally:os.close(fd)
    finally:
        if tmp.exists():tmp.unlink()



def repair_standby_wal_from_primary(c:Coordinator) -> dict:
    """明确的教学“人工授权”：从仍可读取且有完整确认前缀的主机重新建立备机日志。

    绝不尝试从损坏WAL里猜部分事件。不能用于双主分叉自动合并。
    """
    primary=c.rep[c.control['active']]
    backup=c._standby()
    require(primary.up,'源主机不可用，不能据其WAL修复')
    rows=primary.journal.read_records()
    require(len(rows)>=c.control['ack_seq'],'源WAL缺失已确认事件')
    require(primary.identity==backup.identity,'模型身份不一致，禁止修复')
    # 当前教学控制器串行调用；真实集群应停止并隔离备机写入并使用可信离线副本。
    backup.shutdown()
    if backup.journal.snapshot_path.exists():backup.journal.snapshot_path.unlink()
    raw=primary.journal.path.read_bytes()
    _atomic_bytes(backup.journal.path,raw)
    new=Replica(backup.name,backup.folder,c.tok,c.models,c.capacity_blocks,load_existing=True)
    require(new.ledger.to_json()==primary.ledger.to_json(),'重建后的备机逻辑状态不一致')
    c.rep[backup.name]=new
    if backup.name in c.control['quarantined']:
        c.control['quarantined'].remove(backup.name)
        atomic_json(c.control_path,c.control)
    return {'restored_seq':new.ledger.seq,'source_hash':primary.ledger.last_hash,
            'kind':'explicit_replace_from_verified_primary',
            'not_a_split_brain_merge':True}

In [14]:
ALLOWED_FAULTS={
    'primary_down': 'A',
    'standby_down': 'B',
    'standby_torn_wal': 'B',
    'standby_weight_drift': 'B',
    'standby_capacity_pressure': 'B',
}

@dataclass(frozen=True)
class ChaosPlan:
    scope: str='isolated_lab_only'
    hypothesis: str='故障后要么正确恢复，要么明确拒绝冒险恢复'
    max_faults: int=3
    max_tick: int=12
    max_sessions: int=4
    reserve_blocks: int=2
    rpo_budget_ticks: int=0
    rto_budget_ticks: int=16
    allow_loss_override: bool=False
    fault_allowlist: tuple[str,...]=tuple(sorted(ALLOWED_FAULTS))

    def validate(self):
        require(self.scope=='isolated_lab_only','只允许隔离的本地教学演练，禁止指向生产/云资源')
        require(type(self.hypothesis) is str and 8<=len(self.hypothesis)<=160,'演练假设需要明确且有长度上限')
        for key in ('max_faults','max_tick','max_sessions','reserve_blocks','rpo_budget_ticks','rto_budget_ticks'):
            n=getattr(self,key)
            require(type(n) is int and n>=0,'演练预算必须为非负整数，禁止bool')
        require(0<self.max_faults<=5 and 0<self.max_tick<=32 and 0<self.max_sessions<=8,
                '故障、tick或会话数超出教学安全上限')
        require(self.reserve_blocks>=1 and self.rto_budget_ticks>=1,'必须有容量保留与正RTO预算')
        require(type(self.allow_loss_override) is bool,'数据丢失批准必须为显式bool')
        require(type(self.fault_allowlist) is tuple and len(self.fault_allowlist)>0 and
                len(set(self.fault_allowlist))==len(self.fault_allowlist) and
                set(self.fault_allowlist)<=set(ALLOWED_FAULTS),'故障白名单非法')
        return True


class EvidenceTrail:
    """本地hash链：发现意外变更，不提供签名、不可抵赖或远端可信锚点。"""
    FIELDS={'seq','tick','phase','event','data','prev','hash'}
    PHASES={'prepare','inject','gate','observe','repair','verify','stop','final'}
    def __init__(self,path:Path):
        self.path=Path(path)
        require(not self.path.exists(),'证据文件不允许覆盖')
        self.path.write_bytes(b'')
        self.seq=0;self.last=ZERO_HASH;self.last_tick=0

    def append(self,tick:int,phase:str,event:str,data:dict|None=None):
        require(type(tick) is int and self.last_tick<=tick<=1000,'tick不能倒退或越界')
        require(phase in self.PHASES,'未知证据阶段')
        bounded_str(event,80,'事件名')
        data={} if data is None else data
        require(type(data) is dict,'证据数据必须为对象')
        require(len(canonical(data).encode())<=4096,'单条证据过大，禁止保存提示全文或密钥')
        require(not any(any(unsafe in k.lower() for unsafe in ('prompt','token_ids','secret','password','api_key'))
                        for k in data),'证据不记录原始提示、token IDs或凭证')
        core={'seq':self.seq+1,'tick':tick,'phase':phase,'event':event,
              'data':copy.deepcopy(data),'prev':self.last}
        row={**core,'hash':digest(core)}
        with self.path.open('ab') as stream:
            stream.write((canonical(row)+'\n').encode('utf-8'))
            stream.flush();os.fsync(stream.fileno())
        self.seq+=1;self.last=row['hash'];self.last_tick=tick
        return row

    @staticmethod
    def verify(path:Path,expected_head:str|None=None,expected_seq:int|None=None) -> dict:
        raw=Path(path).read_bytes()
        require(bool(raw) and raw.endswith(b'\n'),'证据链为空或有断裂尾行')
        last=ZERO_HASH;seq=0;last_tick=0;phases=[]
        for i,line in enumerate(raw.splitlines(),1):
            try:row=json.loads(line.decode('utf-8'))
            except (UnicodeDecodeError,json.JSONDecodeError) as exc:
                raise LessonError('证据记录不能解码') from exc
            require(type(row) is dict and set(row)==EvidenceTrail.FIELDS,'证据记录字段非法')
            require(type(row['seq']) is int and row['seq']==i and row['prev']==last,
                    '证据序列或前向hash链断裂')
            require(type(row['tick']) is int and last_tick<=row['tick']<=1000,
                    '证据tick不递增')
            require(row['phase'] in EvidenceTrail.PHASES,'未知证据阶段')
            core={k:row[k] for k in row if k!='hash'}
            require(row['hash']==digest(core),'证据内容被改写')
            seq=i;last=row['hash'];last_tick=row['tick'];phases.append(row['phase'])
        if expected_head is not None:require(last==expected_head,'证据head与已封存manifest不一致')
        if expected_seq is not None:require(seq==expected_seq,'证据数量与封存记录不同')
        require(phases[0]=='prepare' and phases[-1]=='final','演练没有完整准备/收尾证据')
        return {'records':seq,'head':last,'last_tick':last_tick,'phases':sorted(set(phases))}


class ChaosGuard:
    """只调度本地对象；不触发真实机器、网络、云实例或持久生产资源故障。"""
    def __init__(self,plan:ChaosPlan,trace:EvidenceTrail):
        plan.validate();self.plan=plan;self.trace=trace
        self.fault_count=0;self.stopped=False;self.tick=0
        self.trace.append(0,'prepare','safety_scope_verified',
                          {'scope':plan.scope,'max_faults':plan.max_faults,
                           'reserve_blocks':plan.reserve_blocks,'manual_loss':plan.allow_loss_override})

    def stop(self,why:str):
        require(type(why) is str and bool(why),'停止原因不能为空')
        if not self.stopped:
            self.stopped=True
            self.tick=max(self.tick,self.trace.last_tick)
            self.trace.append(self.tick,'stop','safety_lever_engaged',{'reason':why})

    def inject(self,cluster:Coordinator,fault:str,target:str,tick:int):
        require(type(tick) is int and tick>=max(self.tick,self.trace.last_tick),'故障tick必须单调')
        self.tick=tick
        if self.stopped or tick>self.plan.max_tick or self.fault_count>=self.plan.max_faults:
            self.stop('safety_budget_or_manual_stop')
            self.trace.append(self.tick,'stop','injection_denied',{'fault':str(fault)[:60]})
            raise LessonError('安全开关或注入预算触发，拒绝新故障')
        require(fault in self.plan.fault_allowlist and ALLOWED_FAULTS.get(fault)==target,
                '注入不在允许的故障/实例目标白名单内')
        require(len(cluster.rep[cluster.control['active']].ledger.sessions)<=self.plan.max_sessions,
                '当前活动会话超过演练范围上限')
        if fault in {'primary_down','standby_down'}:
            cluster.rep[target].shutdown()
        elif fault=='standby_weight_drift':
            model=cluster.rep[target].models['canary']
            params=[p for n,p in model.named_parameters() if n.endswith('.B')]
            require(bool(params),'故障注入未找到LoRA B张量')
            with torch.no_grad():params[0].add_(0.03)
        elif fault=='standby_torn_wal':
            path=cluster.rep[target].journal.path
            with path.open('ab') as stream:
                stream.write(b'{"uncommitted":');stream.flush();os.fsync(stream.fileno())
        elif fault=='standby_capacity_pressure':
            # 仅模拟可用页面预算降低，绝不损坏模型或已分配KV。
            pass
        else:
            raise AssertionError('故障白名单/执行器不一致')
        self.fault_count+=1
        self.trace.append(tick,'inject',fault,{'target':target,'fault_index':self.fault_count})
        return True


def capacity_projection(replica:Replica,available_blocks:int,reserve_blocks:int,
                        planned_new_session_lengths:list[int]|None=None):
    """每会话保守按未来最长上下文估算页数；不计共享前缀收益，避免把优化当容量保证。"""
    require(type(available_blocks) is int and available_blocks>0 and
            type(reserve_blocks) is int and reserve_blocks>=0,'容量必须非负整数且池大小为正')
    new=[] if planned_new_session_lengths is None else planned_new_session_lengths
    require(type(new) is list and all(type(n) is int and 0<n<=replica.pool.cfg.max_length for n in new),
            '新请求长度预算必须为正且不超过上下文')
    block=replica.pool.block_size
    current=0;worst=0;active=0
    for st in replica.ledger.sessions.values():
        if st.terminal is not None:continue
        active+=1
        length=len(st.prompt_ids)+len(st.generated)
        remaining=st.max_new_tokens-len(st.generated)
        require(0<=remaining<=4 and length+remaining<=replica.pool.cfg.max_length,
                '会话剩余token预算不合法')
        current+=math.ceil(length/block)
        worst+=math.ceil((length+remaining)/block)
    worst+=sum(math.ceil(x/block) for x in new)
    total=worst+reserve_blocks
    bytes_per_block=replica.pool.bytes_per_block
    return {'active_sessions':active,'current_required_blocks':current,
            'projected_session_blocks':worst,'emergency_reserved_blocks':reserve_blocks,
            'total_required_blocks':total,'available_blocks':available_blocks,
            'headroom_blocks':available_blocks-total,'allowed':available_blocks>=total,
            'bytes_per_block':bytes_per_block,'projected_reserved_bytes':total*bytes_per_block,
            'physical_capacity_bytes':available_blocks*bytes_per_block,
            'model':'per-session conservative; no prefix share credit; live KV only; excludes model weights'}


def recovery_gate(cluster:Coordinator,plan:ChaosPlan,available_standby_blocks:int,
                  tokens_per_tick:int=32,wal_records_per_tick:int=16) -> dict:
    plan.validate()
    require(type(tokens_per_tick) is int and tokens_per_tick>0 and
            type(wal_records_per_tick) is int and wal_records_per_tick>0,'恢复速率非法')
    old=cluster.rep[cluster.control['active']];new=cluster._standby()
    capacity=capacity_projection(new,available_standby_blocks,plan.reserve_blocks)
    token_count=sum(len(s.prompt_ids)+len(s.generated) for s in new.ledger.sessions.values()
                    if s.terminal is None)
    rto=2+math.ceil(new.ledger.seq/wal_records_per_tick)+math.ceil(token_count/tokens_per_tick)
    lost=max(0,cluster.control['ack_seq']-new.ledger.seq)
    rpo_gap=max(0,cluster.control['ack_tick']-new.ledger.tick)
    issues=[]
    if not new.up:issues.append('standby_unavailable')
    if artifact_identity(new.tok,new.models)!=cluster.model_identity:issues.append('model_identity_drift')
    if old.journal.records[:new.ledger.seq]!=new.journal.records:issues.append('wal_divergence')
    if not capacity['allowed']:issues.append('insufficient_kv_pages')
    if (lost>0 or rpo_gap>plan.rpo_budget_ticks) and not plan.allow_loss_override:
        issues.append('unapproved_rpo_loss')
    if rto>plan.rto_budget_ticks:issues.append('rto_budget_exceeded')
    return {'allowed':not issues,'blockers':issues,'capacity':capacity,
            'ack_gap_events':lost,'rpo_gap_logical_ticks':rpo_gap,
            'predicted_rto_logical_ticks':rto,
            'rto_budget_logical_ticks':plan.rto_budget_ticks,
            'tick_units_only':True,
            'manual_loss_approved':plan.allow_loss_override,
            'note':'planning gate, followed by actual WAL scan, KV replay and failover audit'}


def finish_case(tracer:EvidenceTrail,cluster:Coordinator,report:dict) -> dict:
    audit=cluster.audit()
    require(audit['active_audit']['max_cache_full_logits_diff']<2e-5,'恢复后真实KV/full logits不一致')
    tracer.append(tracer.last_tick,'verify','real_kv_and_control_audited',
                  {'epoch':audit['epoch'],'active':audit['active'],
                   'ack_seq':audit['ack_seq'],
                   'kv_live_blocks':audit['active_audit']['kv_live_blocks']})
    result={**report,'final_control':{'epoch':cluster.control['epoch'],
             'active':cluster.control['active'],'ack_seq':cluster.control['ack_seq'],
             'active_ledger_hash':cluster.rep[cluster.control['active']].ledger.last_hash},
            'kv_max_diff':audit['active_audit']['max_cache_full_logits_diff'],
            'physical_kv_live_blocks':audit['active_audit']['kv_live_blocks']}
    tracer.append(tracer.last_tick,'final','case_completed',{'report_hash':digest(result)})
    return result

In [15]:
safe_plan=ChaosPlan()
print('计划通过:',safe_plan.validate())
print('白名单:',safe_plan.fault_allowlist)
for target in ['production','isolated_lab_only']:
    try:
        allowed=ChaosPlan(scope=target).validate()
        print(target,'=>',allowed)
    except LessonError as exc:
        print(target,'=> 被拒绝：',str(exc))

计划通过: True
白名单: ('primary_down', 'standby_capacity_pressure', 'standby_down', 'standby_torn_wal', 'standby_weight_drift')
production => 被拒绝： 只允许隔离的本地教学演练，禁止指向生产/云资源
isolated_lab_only => True


## 7. 灾备容量：保守，不借用未证实的缓存复用收益

本课容量预算按每个仍活跃会话的**最长剩余输出**估算需求：

\[B_{need}=\sum_i\left\lceil\frac{L_i+R_i}{B}\right\rceil
+\sum_j\left\lceil\frac{L^{new}_j}{B}\right\rceil+B_{reserve}.\]

其中：$L_i$ 是当前 token 长度，$R_i$ 是该请求还允许生成的 token 数，$B$ 是每物理块的 token 容量。先逐请求向上取整、再相加；**不能先把所有请求长度相加再向上取整**，否则跨请求“共享尾部空位”，会低估真实块数。

它保守地不计共享前缀节省，也不包含模型权重和运行时临时内存；真实 CUDA 容量仍需设备上的独立测量。

In [16]:
print('手算：活跃会话(5+2)与(8+1)，新增长度5，块大小4，保留2块。')
print('每会话页数：2+3+2+2 = 9')
print('足够9块:',answer_a([(5,2),(8,1)],4,[5],2,9) if 'answer_a' in globals() else '参考答案在后面')

手算：活跃会话(5+2)与(8+1)，新增长度5，块大小4，保留2块。
每会话页数：2+3+2+2 = 9
足够9块: 参考答案在后面


## 8. “恢复计划被允许”并不等于“实际恢复成功”

Recovery Gate 检查：目标副本可用、完整模型身份、WAL 前缀、KV 容量、已 ACK 数据缺口、逻辑 RPO 和逻辑 RTO。所有项通过后，才执行 `Coordinator.promote()` 的**真实持久日志冷加载与KV重建**，并再次审计 logits 和页池。

容量门槛不足时，本课不仅输出一个计算出来的 `False`：还会创建实际太小的 KV 页池并尝试从 WAL 重建，确认真的会遭遇资源不足。

RTO 仅使用教学公式：

\[2+\lceil verified\_WAL / records\_per\_tick\rceil+\lceil rebuilt\_tokens / tokens\_per\_tick\rceil.\]

这不是墙钟耗时，更不是线上 SLO 保证；若需量产评测必须覆盖探测、网络、存储、排队、实例启动和客户端恢复的实测路径。

In [17]:
print('RPO：已ACK=10，备份=8，ACK tick=10，备份tick=8')
print('默认不批准数据丢失，应拒绝自动接管。')
print('参考数学例子将在后面的练习区运行。')

RPO：已ACK=10，备份=8，ACK tick=10，备份tick=8
默认不批准数据丢失，应拒绝自动接管。
参考数学例子将在后面的练习区运行。


## 9. 逐个阅读九个可重复的实际演练

| 场景 | 注入或边界 | 要验收的关键结论 |
|---|---|---|
| `steady_state` | 无故障对照 | 同步日志/真实KV一致，终态释放 |
| `sync_primary_loss` | 主机下线 | 零ACK损失、旧epoch围栏、幂等重试 |
| `sync_standby_outage` | 备机先下线 | 不能假装sync ACK，恢复后同键确认原事件 |
| `two_replicas_down` | 两台都下线 | 自动切主失败，紧急停止后才能可信重建 |
| `standby_kv_shortfall` | 备机保留容量不足 | 保守预算否决＋真实页池OOM，再补容量恢复 |
| `async_rpo_gap` | 异步落后且主机故障 | 未授权已ACK损失禁止切主；可信源补齐WAL |
| `async_loss_authorized` | 同上，显式授权 | 即使授权也要报告丢失事件并隔离旧主 |
| `model_drift_compound` | LoRA 漂移＋主机故障 | 模型指纹拒绝自动接管，重建后才可推广 |
| `torn_wal_compound` | 备份WAL断尾＋主机故障 | 拒绝静默截断，只用可读可信主盘显式重建 |

“检测到预期故障并安全拒绝”也可能是该场景验收成功。不能把它写成服务在两台都坏时仍可用。

In [18]:
def make_drill_cluster(root:Path,name:str,tok,models,policy='sync'):
    case=Path(root)/name
    case.mkdir(parents=True,exist_ok=False)
    return Coordinator(case,tok,models,policy=policy,seed=8201,capacity_blocks=64)


def create_case(root,name,tok,models,policy='sync',plan:ChaosPlan|None=None):
    cluster=make_drill_cluster(root,name,tok,models,policy)
    tracer=EvidenceTrail(cluster.root/'chaos_evidence.jsonl')
    guard=ChaosGuard(plan or ChaosPlan(),tracer)
    return cluster,tracer,guard


def case_healthy(root,tok,models):
    c,t,g=create_case(root,'steady_state',tok,models)
    first=c.start(1,'team','one','stable','hello',max_new_tokens=4)
    second=c.start(1,'team','two','canary','answer 42',max_new_tokens=4)
    _advance(c,first['rid'],1)
    before={name:c.rep[name].audit() for name in ('A','B')}
    readiness=recovery_gate(c,g.plan,64)
    require(readiness['allowed'] and before['A']['logical_seq']==before['B']['logical_seq'],
            '健康同步集群不应阻断恢复规划')
    t.append(1,'gate','healthy_capacity_baseline',
             {'required_blocks':readiness['capacity']['total_required_blocks'],
              'logical_ack':c.control['ack_seq']})
    for rid in (first['rid'],second['rid']):
        while c.rep['A'].ledger.sessions[rid].terminal is None:_advance(c,rid,1)
    require(all(r.pool.usage()['live_blocks']==0 for r in c.rep.values()),
            '真实终止后各副本KV必须回收')
    report={'hypothesis':'无故障时同步确认且两个副本KV最终释放',
            'kind':'healthy_control','healthy_acks':c.control['ack_seq'],
            'all_physical_kv_released':True,'readiness':readiness,
            'real_kv_full_diff_max_before':max(x['max_cache_full_logits_diff'] for x in before.values())}
    return finish_case(t,c,report)


def case_sync_primary_loss(root,tok,models):
    c,t,g=create_case(root,'sync_primary_loss',tok,models)
    item=c.start(1,'team','resume','stable','hello',max_new_tokens=4)
    _advance(c,item['rid'],1)
    gate=recovery_gate(c,g.plan,64)
    require(gate['allowed'],'零RPO同步热备容量不足')
    t.append(1,'gate','pre_fault_readiness',{'capacity_ok':True,'rto':gate['predicted_rto_logical_ticks']})
    ack_before=c.control['ack_seq']
    g.inject(c,'primary_down','A',2)
    failover=c.promote(1,'B',rpo_budget_ticks=0,rto_budget_ticks=16)
    require(failover['lost_acked_events']==0 and c.control['ack_seq']==ack_before,
            '同步模式灾备丢失已确认记录')
    t.append(3,'observe','failover_verified',{'new_epoch':c.control['epoch'],'lost_acks':0})
    duplicate=c.step(2,item['rid'],0)
    require(not duplicate['new_event'],'切主后原有token重试被错误重新生成')
    stale=False
    try:c.start(1,'team','stale','stable','hello')
    except LessonError:stale=True
    require(stale,'旧epoch仍能写入')
    # 可恢复的只是模拟旧主磁盘；先恢复备用实例，随后才能继续sync ACK。
    c.rep['A']=Replica('A',c.root/'A',tok,models,load_existing=True)
    while c.rep['B'].ledger.sessions[item['rid']].terminal is None:_advance(c,item['rid'],1)
    require(all(r.pool.usage()['live_blocks']==0 for r in c.rep.values()),'完成后KV未释放')
    t.append(4,'verify','idempotency_and_epoch_fence',
             {'duplicate_rejected':True,'old_epoch_rejected':stale})
    return finish_case(t,c,{'kind':'sync_failover','hypothesis':'零丢失、旧主围栏和幂等重试',
       'ack_before':ack_before,'failover':failover,'idempotent_retry':True,
       'old_primary_fenced':True,'zero_kv_after_terminal':True})


def case_sync_replica_down(root,tok,models):
    c,t,g=create_case(root,'sync_standby_outage',tok,models)
    g.inject(c,'standby_down','B',1)
    refused=False
    try:c.start(1,'team','ambig','stable','hello',max_new_tokens=4)
    except LessonError:refused=True
    require(refused and c.control['ack_seq']==0 and c.rep['A'].ledger.seq==1,
            'sync ACK必须等待两个实例完成WAL持久化')
    t.append(2,'observe','ack_withheld',{'ack_seq':0,'primary_durable_seq':1})
    c.rep['B']=Replica('B',c.root/'B',tok,models,load_existing=True)
    retry=c.start(1,'team','ambig','stable','hello',max_new_tokens=4)
    require(not retry['new_event'] and c.control['ack_seq']==1 and
            c.rep['B'].ledger.seq==1,'同一幂等键应确认原WAL且修复副本')
    t.append(3,'repair','standby_disk_rebuilt',{'duplicate_create':False,'ack_seq':1})
    return finish_case(t,c,{'kind':'sync_ack_gate','hypothesis':'备机故障不能假装sync已确认',
             'unacked_primary_write':refused,'retried_original_event':True,
             'ack_after_replica_repair':1})


def case_double_failure(root,tok,models):
    c,t,g=create_case(root,'two_replicas_down',tok,models)
    c.start(1,'team','dual','stable','hello')
    g.inject(c,'standby_down','B',1)
    g.inject(c,'primary_down','A',2)
    blocked=False
    try:c.promote(1,'B')
    except LessonError:blocked=True
    require(blocked and c.control['epoch']==1,'双实例不可用时禁止虚假切主成功')
    t.append(3,'gate','dual_outage_fail_closed',{'promote_blocked':blocked})
    g.stop('两台实例均不可用，需人工恢复磁盘与验证')
    post_stop_denied=False
    try:g.inject(c,'primary_down','A',4)
    except LessonError:post_stop_denied=True
    require(post_stop_denied,'安全开关不允许再注入')
    c.rep['B']=Replica('B',c.root/'B',tok,models,load_existing=True)
    report=c.promote(1,'B',rpo_budget_ticks=0,rto_budget_ticks=16)
    require(report['lost_acked_events']==0,'从可信备机磁盘恢复不得丢失确认日志')
    t.append(5,'repair','trusted_standby_disk_restored',{'epoch':2,'restored':True})
    return finish_case(t,c,{'kind':'compound_two_down','hypothesis':'双机故障必须先停止试验再重建',
                'automatic_failover_blocked':blocked,'kill_switch_denied_next_fault':True,
                'recovered_from_simulated_durable_disk':True,'lost_acks':0})


def case_kv_capacity(root,tok,models):
    c,t,g=create_case(root,'standby_kv_shortfall',tok,models)
    ids=[]
    for j in range(3):
        ids.append(c.start(1,'team',f'capacity_{j}','stable','answer 42',max_new_tokens=4)['rid'])
    g.inject(c,'standby_capacity_pressure','B',1)
    too_small=recovery_gate(c,g.plan,4)
    require(not too_small['allowed'] and 'insufficient_kv_pages' in too_small['blockers'],
            '容量不足必须在自动接管之前被拒绝')
    t.append(2,'gate','capacity_denied',{'required':too_small['capacity']['total_required_blocks'],
              'available':4})
    # 不仅模拟计算：强制用真实4个页重建多条会话，必须触发物理KV OOM。
    real_oom=False
    try:Replica('B',c.root/'B',tok,models,capacity_blocks=4,load_existing=True)
    except LessonError:real_oom=True
    require(real_oom,'真实KV重建竟然在小于需求的页池中成功')
    sufficient=recovery_gate(c,g.plan,64)
    require(sufficient['allowed'],'扩大灾备页池后仍无法满足保守门槛')
    g.inject(c,'primary_down','A',3)
    promoted=c.promote(1,'B',rpo_budget_ticks=0,rto_budget_ticks=16)
    require(promoted['lost_acked_events']==0,'容量修复后不应丢失同步确认')
    t.append(4,'verify','full_recovery_after_capacity_fix',
             {'recomputed_tokens':promoted['recovered_tokens_recomputed'],
              'real_oom_blocked':True})
    return finish_case(t,c,{'kind':'kv_capacity','hypothesis':'页池耗尽应在切主前被拒绝并实测验证',
               'capacity_denial':too_small,'sufficient_gate':sufficient,
               'physical_kv_oom_verified':True,'failover_after_capacity_fix':promoted})


def case_async_loss_gate(root,tok,models):
    c,t,g=create_case(root,'async_rpo_gap',tok,models,policy='async')
    item=c.start(1,'team','async','stable','hello',max_new_tokens=4)
    require(c.replicate_pending()==1,'异步开始事件未预先复制')
    _advance(c,item['rid'],2)
    require(c.control['ack_seq']-c.rep['B'].ledger.seq==2,'未产生预期异步RPO缺口')
    g.inject(c,'primary_down','A',1)
    denied=recovery_gate(c,g.plan,64)
    require(not denied['allowed'] and 'unapproved_rpo_loss' in denied['blockers'],
            '缺失已确认事件仍自动切主')
    t.append(2,'gate','unapproved_loss_blocked',
             {'lost_ack_events':denied['ack_gap_events'],'override':False})
    blocked=False
    try:c.promote(1,'B',rpo_budget_ticks=0)
    except LessonError:blocked=True
    require(blocked and c.control['epoch']==1,'RPO缺口未拒绝切主')
    # 人工选择读取原主机仍可访问的可信本地磁盘，补齐 WAL；不是假定彻底丢盘还可无损。
    c.rep['A']=Replica('A',c.root/'A',tok,models,load_existing=True)
    copied=c.replicate_pending()
    require(copied==2,'未从原主可信持久文件补齐两个事件')
    c.rep['A'].shutdown()
    allowed=recovery_gate(c,g.plan,64)
    require(allowed['allowed'],'日志补齐后不应继续报告RPO缺口')
    report=c.promote(1,'B',rpo_budget_ticks=0)
    require(report['lost_acked_events']==0,'日志补齐后仍然丢失ACK')
    t.append(3,'repair','trusted_primary_wal_tail_copied',
             {'copied_events':copied,'lost_ack_events_after':0})
    return finish_case(t,c,{'kind':'async_rpo_denial','hypothesis':'没有明确授权不得丢失异步已确认事件',
               'strict_auto_denied':blocked,'initial_denial':denied,
               'verified_primary_disk_readable':True,'copied_durable_tail':copied,
               'after_repair_loss':report['lost_acked_events']})


def case_async_explicit_loss(root,tok,models):
    plan=ChaosPlan(allow_loss_override=True)
    c,t,g=create_case(root,'async_loss_authorized',tok,models,policy='async',plan=plan)
    item=c.start(1,'team','loss','stable','hello',max_new_tokens=4)
    c.replicate_pending()
    _advance(c,item['rid'],1)
    require(c.control['ack_seq']-c.rep['B'].ledger.seq==1,'未产生一个丢失事件')
    g.inject(c,'primary_down','A',1)
    allowed=recovery_gate(c,g.plan,64)
    require(allowed['allowed'] and allowed['manual_loss_approved'],'人工授权未明确打开')
    t.append(2,'gate','manual_loss_override',{'expected_lost_ack':1,'approval':True})
    report=c.promote(1,'B',allow_data_loss=True,rpo_budget_ticks=0,rto_budget_ticks=16)
    require(report['lost_acked_events']==1 and report['data_loss_override'],
            '丢失确认信息未被显式记录')
    require(c.control['quarantined']==['A'],'丢失已ACK事件后旧主必须隔离')
    t.append(3,'verify','loss_declared_and_primary_quarantined',
             {'lost_acks':1,'old_primary_quarantined':True})
    return finish_case(t,c,{'kind':'async_loss_approved','hypothesis':'显式授权也不能把已ACK数据丢失写成零',
        'loss_approval':True,'lost_acked_events':1,
        'client_reconciliation_required':True,'old_primary_quarantined':True,
        'failover':report})


def case_weight_drift(root,tok,models):
    c,t,g=create_case(root,'model_drift_compound',tok,models)
    c.start(1,'team','drift','canary','hello')
    g.inject(c,'standby_weight_drift','B',1)
    gate=recovery_gate(c,g.plan,64)
    require(not gate['allowed'] and 'model_identity_drift' in gate['blockers'],
            '模型权重漂移未触发门槛')
    g.inject(c,'primary_down','A',2)
    blocked=False
    try:c.promote(1,'B')
    except LessonError:blocked=True
    require(blocked and c.control['epoch']==1,'复合故障绕开了模型身份校验')
    t.append(3,'gate','weight_drift_blocked',{'reason':'model_identity_drift'})
    # 从与控制版本绑定的可信本地artifact重建，不能在漂移张量上继续提供服务。
    c.rep['B'].shutdown()
    c.rep['B']=Replica('B',c.root/'B',tok,models,load_existing=True)
    require(recovery_gate(c,g.plan,64)['allowed'],'重建后仍没有满足恢复门槛')
    report=c.promote(1,'B')
    t.append(4,'repair','adapter_identity_restored',{'new_epoch':2,'lost_acks':0})
    return finish_case(t,c,{'kind':'drift_plus_primary_down','hypothesis':'失败副本的模型身份不可信，禁止直接接管',
       'promotion_blocked_before_repair':blocked,'verified_artifact_reloaded':True,
       'no_loss_after_repair':report['lost_acked_events']==0})


def case_torn_wal(root,tok,models):
    c,t,g=create_case(root,'torn_wal_compound',tok,models)
    c.start(1,'team','torn','stable','hello')
    g.inject(c,'standby_torn_wal','B',1)
    g.inject(c,'primary_down','A',2)
    blocked=False
    try:c.promote(1,'B')
    except LessonError:blocked=True
    require(blocked and c.control['active']=='A','断裂WAL不允许自动修复并接管')
    t.append(3,'gate','torn_wal_recovery_blocked',{'automatic_repair':False})
    # 仅演示“原主数据盘仍可读”的可修复情况；若源也损坏，必须保留人工停机。
    c.rep['A']=Replica('A',c.root/'A',tok,models,load_existing=True)
    repair=repair_standby_wal_from_primary(c)
    c.rep['A'].shutdown()
    require(recovery_gate(c,g.plan,64)['allowed'],'人工重建后的备份依然无法恢复')
    report=c.promote(1,'B')
    t.append(4,'repair','standby_wal_replaced_explicitly',
             {'source_head':repair['source_hash'],'lost_acks':report['lost_acked_events']})
    return finish_case(t,c,{'kind':'wal_torn_compound','hypothesis':'断裂WAL必须拒绝自动截断，明确验证可信来源',
       'corrupt_recovery_blocked':blocked,'repair_kind':repair['kind'],
       'source_was_readable':True,'recovered_with_zero_loss':report['lost_acked_events']==0})

## 10. 运行与保存证据

演练会创建一次性目录，每个子案例保存：

```text
models.pt / control.json    真实模型权重和持久控制指针
A/events.jsonl             A实例的严格WAL
B/events.jsonl             B实例的严格WAL
chaos_evidence.jsonl       受控注入与恢复阶段的哈希链
result.json                场景结论、阻断原因和恢复状态
```

顶层 `scenarios.json`、`evidence_index.json` 与 `manifest.json` 保存完成时的报告摘要；`reload.json` 是**丢弃内存对象后，从磁盘重新加载全部九个案例**的审计。裸SHA-256与本地封存不提供攻击者篡改防护，强威胁模型需要外部可信锚点。

每个WAL仍可能含可还原文本的token ID，**不要向公网分享生成的演练输出目录**。本课只要求下载两个源码文件。

In [19]:
SCENARIOS={
    'steady_state':case_healthy,
    'sync_primary_loss':case_sync_primary_loss,
    'sync_standby_outage':case_sync_replica_down,
    'two_replicas_down':case_double_failure,
    'standby_kv_shortfall':case_kv_capacity,
    'async_rpo_gap':case_async_loss_gate,
    'async_loss_authorized':case_async_explicit_loss,
    'model_drift_compound':case_weight_drift,
    'torn_wal_compound':case_torn_wal,
}


def reload_run(folder:Path) -> dict:
    """磁盘上独立重建模型/真实KV；对每个快照、WAL、会话状态和证据hash验证。"""
    folder=Path(folder)
    manifest=json.loads((folder/'manifest.json').read_text(encoding='utf-8'))
    require(manifest['version']==VERSION and manifest['status'] in {'running','completed'},
            'manifest的版本或状态错误')
    require(manifest['scenario_names']==list(SCENARIOS),'演练清单与运行版本不一致')
    reports=json.loads((folder/'scenarios.json').read_text(encoding='utf-8'))
    index=json.loads((folder/'evidence_index.json').read_text(encoding='utf-8'))
    require(set(reports)==set(index)==set(SCENARIOS),'演练报告或证据索引缺失/增添')
    require(manifest['scenario_report_hash']==digest(reports) and
            manifest['evidence_index_hash']==digest(index),
            '报告/证据索引与封存manifest不一致')
    audit={}
    for name in SCENARIOS:
        expected=index[name]
        require(type(expected) is dict and
                set(expected)=={'trace_hash','trace_records','report_hash'},
                '证据索引字段错误')
        result=reports[name]
        saved=json.loads((folder/name/'result.json').read_text(encoding='utf-8'))
        require(result==saved and digest(result)==expected['report_hash'],
                '独立情境结果被篡改或复制错位')
        trace=EvidenceTrail.verify(folder/name/'chaos_evidence.jsonl',
                        expected['trace_hash'],expected['trace_records'])
        # Fresh objects from trusted locally generated weights/WAL; do not reuse original in-memory KV.
        restored=restore_controller(folder/name)
        require(restored.model_identity==manifest['model_identity'],
                '场景权重/模板身份与统一演练基准不符')
        live=restored.rep[restored.control['active']]
        proof=result['final_control']
        require(proof=={'epoch':restored.control['epoch'],
                        'active':restored.control['active'],
                        'ack_seq':restored.control['ack_seq'],
                        'active_ledger_hash':live.ledger.last_hash},
                '恢复控制指针/确认水位与报告不一致')
        check=restored.audit()
        require(check['active_audit']['max_cache_full_logits_diff']<2e-5,
                '新进程重建的KV与全前向不同')
        require(check['active_audit']['kv_live_blocks']==result['physical_kv_live_blocks'],
                '恢复后物理KV占用与报告不一致')
        audit[name]={'trace_records':trace['records'],'trace_hash_matches':True,
                     'restored_epoch':proof['epoch'],'restored_ack_seq':proof['ack_seq'],
                     'max_kv_full_logits_error':check['active_audit']['max_cache_full_logits_diff'],
                     'physical_kv_blocks':check['active_audit']['kv_live_blocks'],
                     'fresh_model_loaded_from_disk':True}
        for replica in restored.rep.values():replica.shutdown()
    return {'version':VERSION,'scenarios':audit,
            'all_nine_rebuilt_independently':len(audit)==9,
            'no_wall_clock_rto_measurement':True,
            'model':'toy_causal_transformer_with_real_incremental_kv'}


def run_experiment(output_dir:Path=Path('lesson_82_outputs'),seed=8201) -> Path:
    require(type(seed) is int and 0<=seed<2**31,'seed必须为非负整数')
    torch.set_num_threads(1);seed_all(seed)
    tok,models=build_servable_models(seed)
    out=Path(output_dir)/datetime.now(timezone.utc).strftime('run_%Y%m%dT%H%M%S_%fZ')
    out.mkdir(parents=True,exist_ok=False)
    manifest={'version':VERSION,'status':'running','seed':seed,
              'torch':str(torch.__version__),'python':platform.python_version(),
              'device':'cpu','model_identity':artifact_identity(tok,models),
              'scenario_names':list(SCENARIOS),'sources':SOURCES,
              'scope':'isolated_lab_only; no host/cloud/network operations',
              'limitations':['single_process_simulated_two_disks_not_2_hosts',
                    'logical_ticks_not_real_time_RTO_RPO',
                    'toy_random_CausalLM_not_pretrained_serving_speed',
                    'no_consensus_no_real_distributed_fencing',
                    'SHA256_not_signature_or_external_trust_anchor',
                    'no_PII_protection_or_encryption_of_local_WAL',
                    'KV_on_CPU_not_GPU_capacity_test',
                    'physical_KV_recomputed_not_persisted']}
    write_json(out/'manifest.json',manifest)
    reports={};index={}
    try:
        for name,fn in SCENARIOS.items():
            r=fn(out,tok,models)
            require(r['kind'] and r['final_control']['epoch']>=1,
                    '场景没有有效的结论与恢复点')
            reports[name]=r
            write_json(out/name/'result.json',r)
            trace=EvidenceTrail.verify(out/name/'chaos_evidence.jsonl')
            index[name]={'trace_hash':trace['head'],'trace_records':trace['records'],
                         'report_hash':digest(r)}
            print(name,'完成；证据',trace['records'],'条',flush=True)
        write_json(out/'scenarios.json',reports)
        write_json(out/'evidence_index.json',index)
        manifest['scenario_report_hash']=digest(reports)
        manifest['evidence_index_hash']=digest(index)
        write_json(out/'manifest.json',manifest)
        replay=reload_run(out)
        write_json(out/'reload.json',replay)
        manifest['status']='completed'
        write_json(out/'manifest.json',manifest)
        print('九项演练与磁盘冷恢复检查完成：',out,flush=True)
        return out
    except Exception as exc:
        manifest['status']='failed';manifest['error']=type(exc).__name__+': '+str(exc)
        write_json(out/'manifest.json',manifest)
        raise

In [20]:
print('场景总数:',len(SCENARIOS))
print('场景名:',list(SCENARIOS))
print('正常案例也保存日志；各场景使用独立磁盘目录。')

场景总数: 9
场景名: ['steady_state', 'sync_primary_loss', 'sync_standby_outage', 'two_replicas_down', 'standby_kv_shortfall', 'async_rpo_gap', 'async_loss_authorized', 'model_drift_compound', 'torn_wal_compound']
正常案例也保存日志；各场景使用独立磁盘目录。


## 11. 固定检查先于完整演练

检查覆盖：范围白名单、安全停止、证据哈希链、尾行截断、参数身份、同步 ACK、实际双副本 KV、页级容量边界、保守预留、逻辑 RPO/RTO、真实 OOM、显式数据损失与旧主围栏。

所有场景都使用真实物理 KV，不是只改 JSON 里的布尔值。检查只证明**本课定义的机制**，不是双区域生产系统可靠性、云安全认证或GPU容量承诺。

In [21]:
class Checks:
    def __init__(self):self.names=[]
    def ok(self,label,condition):
        if not bool(condition):raise AssertionError(label)
        self.names.append(label)
    def equal(self,label,actual,expected):self.ok(label,actual==expected)
    def near(self,label,actual,expected,tol=2e-5):
        self.ok(label,abs(float(actual)-float(expected))<=tol)
    def rejects(self,label,func):
        try:func()
        except (LessonError,ValueError):self.names.append(label)
        else:raise AssertionError(label+': 预期拒绝但成功')


def run_checks() -> dict:
    torch.set_num_threads(1);seed_all(8201)
    x=Checks();plan=ChaosPlan()
    x.ok('默认演练范围隔离',plan.scope=='isolated_lab_only')
    x.ok('默认计划合法',plan.validate())
    x.ok('默认拒绝ACK损失授权',not plan.allow_loss_override)
    x.equal('默认至少两页预留',plan.reserve_blocks,2)
    x.ok('注入白名单严格限定目标',ALLOWED_FAULTS['primary_down']=='A' and
         all(t=='B' for k,t in ALLOWED_FAULTS.items() if k!='primary_down'))
    x.rejects('生产范围被拒绝',lambda:ChaosPlan(scope='production').validate())
    x.rejects('云资源范围被拒绝',lambda:ChaosPlan(scope='aws_ec2').validate())
    x.rejects('非法故障白名单拒绝',lambda:ChaosPlan(fault_allowlist=('prod_shutdown',)).validate())
    x.rejects('白名单重复拒绝',lambda:ChaosPlan(fault_allowlist=('primary_down','primary_down')).validate())
    x.rejects('布尔故障预算拒绝',lambda:ChaosPlan(max_faults=True).validate())
    x.rejects('零故障预算拒绝',lambda:ChaosPlan(max_faults=0).validate())
    x.rejects('故障预算过大拒绝',lambda:ChaosPlan(max_faults=6).validate())
    x.rejects('最大逻辑tick越界',lambda:ChaosPlan(max_tick=33).validate())
    x.rejects('会话上限过大拒绝',lambda:ChaosPlan(max_sessions=9).validate())
    x.rejects('零KV安全余量拒绝',lambda:ChaosPlan(reserve_blocks=0).validate())
    x.rejects('零RTO预算拒绝',lambda:ChaosPlan(rto_budget_ticks=0).validate())
    x.rejects('隐式loss授权类型拒绝',lambda:ChaosPlan(allow_loss_override=1).validate())
    tok=ByteMergeChatTokenizer()
    text='hello';encoded=tok.apply_chat_template([{'role':'user','content':text}],tokenize=True,
                                               add_generation_prompt=True)
    x.ok('真正的chat模板产生token',isinstance(encoded,list) and len(encoded)>5)
    x.ok('固定模板不将用户控制字符当结构token',len(tok.contract())>0)
    seed_all(8201);tok,models=build_servable_models(8201)
    x.ok('固定双版本模型',set(models)=={'stable','canary'})
    x.ok('两个模型权重身份可区分',state_hash(models['stable'].state_dict())!=state_hash(models['canary'].state_dict()))
    x.ok('推理模型全冻结',all(not p.requires_grad for m in models.values() for p in m.parameters()))
    cfg=next(iter(models.values())).cfg
    x.ok('真实多层注意力模型',cfg.layers>=1 and cfg.heads>=1)
    with tempfile.TemporaryDirectory() as tmp:
        root=Path(tmp)
        c,t,g=create_case(root,'guard',tok,models)
        x.ok('证据准备记录持久化',t.seq==1 and t.last!=ZERO_HASH)
        x.rejects('禁止故障目标混用',lambda:g.inject(c,'standby_down','A',1))
        x.rejects('禁止未知类型故障',lambda:g.inject(c,'remove_all_disks','B',1))
        x.ok('拒绝不计故障预算',g.fault_count==0)
        x.rejects('故障注入tick不能倒退',lambda:g.inject(c,'standby_down','B',-1))
        x.ok('范围拒绝不改变WAL',c.rep['B'].ledger.seq==0)
        plan_limits=ChaosPlan(max_faults=1,max_tick=1)
        c2,tr,g2=create_case(root,'guard_budget',tok,models,plan=plan_limits)
        g2.inject(c2,'standby_down','B',1)
        x.equal('一次真实关闭备机',c2.rep['B'].up,False)
        x.equal('故障预算计数正确',g2.fault_count,1)
        x.rejects('超过故障预算拒绝',lambda:g2.inject(c2,'primary_down','A',2))
        x.equal('超限安全开关启动',g2.stopped,True)
        x.ok('超限不会杀掉额外主机',c2.rep['A'].up)
        x.rejects('已停止不能重启注入',lambda:g2.inject(c2,'primary_down','A',3))
        x.ok('安全拒绝不更改ACK',c2.control['ack_seq']==0)
        x.ok('每条证据有有效hash',tr.last!=ZERO_HASH)
        tr.append(3,'final','case_completed',{'status':'guard_test'})
        verified=EvidenceTrail.verify(tr.path,tr.last,tr.seq)
        x.equal('证据独立核对序列数',verified['records'],tr.seq)
        x.equal('证据独立核对head',verified['head'],tr.last)
        x.rejects('证据head错配拒绝',lambda:EvidenceTrail.verify(tr.path,'f'*64))
        x.rejects('证据count错配拒绝',lambda:EvidenceTrail.verify(tr.path,expected_seq=tr.seq+1))
        before=tr.path.read_bytes()
        tr.path.write_bytes(before.replace(b'standby_down',b'standby_dowN',1))
        x.rejects('证据payload被改动检测',lambda:EvidenceTrail.verify(tr.path))
        tr.path.write_bytes(before[:-1])
        x.rejects('证据断裂尾行检测',lambda:EvidenceTrail.verify(tr.path))
        tr.path.write_bytes(before)
        x.equal('恢复后原证据仍可核对',EvidenceTrail.verify(tr.path)['records'],tr.seq)
        x.rejects('证据不记录token_ids',lambda:tr.append(4,'observe','bad',{'token_ids':[1,2,3]}))
        x.rejects('证据不记录凭证',lambda:tr.append(4,'observe','bad',{'api_key':'abc'}))
        x.rejects('证据不允许tick倒退',lambda:tr.append(1,'observe','bad'))
        x.rejects('证据阶段需在白名单',lambda:tr.append(4,'unknown','bad'))
        x.rejects('证据单行体积受限',lambda:tr.append(4,'observe','bad',{'data':'0'*5000}))
        # 控制器真实同步事件、两份物理KV
        started=c.start(1,'team','one','stable','hello')
        x.equal('sync首先同步复制create',c.rep['A'].ledger.seq,c.rep['B'].ledger.seq)
        x.equal('sync确认日志水位',c.control['ack_seq'],1)
        audits={key:r.audit() for key,r in c.rep.items()}
        x.ok('两侧KV真实占页',all(r['kv_live_blocks']>0 for r in audits.values()))
        x.ok('两侧KV逐token等价full',all(r['max_cache_full_logits_diff']<2e-5 for r in audits.values()))
        cap=capacity_projection(c.rep['B'],64,2)
        x.equal('容量需求含独立保留',cap['total_required_blocks'],cap['projected_session_blocks']+2)
        x.ok('容量足够允许',cap['allowed'])
        x.equal('页池每块字节与KV模型一致',cap['projected_reserved_bytes'],cap['total_required_blocks']*cap['bytes_per_block'])
        x.ok('容量显示不等于模型权重',cap['model'].startswith('per-session'))
        x.ok('容量所需不能小于当前占用',cap['projected_session_blocks']>=cap['current_required_blocks'])
        exact=cap['total_required_blocks']
        x.equal('容量恰好边界允许',capacity_projection(c.rep['B'],exact,2)['allowed'],True)
        x.equal('容量差一页拒绝',capacity_projection(c.rep['B'],exact-1,2)['allowed'],False)
        x.equal('添加新会话需额外页',capacity_projection(c.rep['B'],64,2,[9])['projected_session_blocks'],
                cap['projected_session_blocks']+math.ceil(9/c.rep['B'].pool.block_size))
        x.rejects('拒绝负可用页数',lambda:capacity_projection(c.rep['B'],-1,2))
        x.rejects('拒绝bool容量',lambda:capacity_projection(c.rep['B'],True,2))
        x.rejects('拒绝负reserve',lambda:capacity_projection(c.rep['B'],64,-1))
        x.rejects('拒绝零新会话上下文',lambda:capacity_projection(c.rep['B'],64,2,[0]))
        x.rejects('拒绝超长新会话',lambda:capacity_projection(c.rep['B'],64,2,[cfg.max_length+1]))
        allowed=recovery_gate(c,plan,64)
        x.ok('健康sync门槛允许接管',allowed['allowed'])
        x.equal('恢复预测明确标为tick',allowed['tick_units_only'],True)
        x.equal('零RPO缺口',allowed['ack_gap_events'],0)
        x.equal('正常无阻断器',allowed['blockers'],[])
        x.ok('RTO预测有基础成本',allowed['predicted_rto_logical_ticks']>=2)
        x.equal('容量不足反映在阻断器',recovery_gate(c,plan,1)['blockers'],['insufficient_kv_pages'])
        strict=ChaosPlan(rto_budget_ticks=1)
        x.ok('过短RTO触发拒绝','rto_budget_exceeded' in recovery_gate(c,strict,64)['blockers'])
        x.rejects('恢复速率零非法',lambda:recovery_gate(c,plan,64,tokens_per_tick=0))
        x.rejects('WAL重放速率零非法',lambda:recovery_gate(c,plan,64,wal_records_per_tick=0))
        # 确保真实模型冻结前向和token序列不可被WAL篡改
        bad=c.rep['B'].journal.records[0].copy();bad['hash']='0'*64
        x.rejects('WAL链hash无法随意替换',lambda:c.rep['B'].journal.ledger.apply(bad))
        x.ok('当前模型指纹可重复计算',artifact_identity(tok,models)==c.model_identity)
        t.append(4,'final','case_completed',{'status':'guard_end'})
        # 上下文长度与物理页池：实际小页面一定发生容量不足。
        little=c.rep['B'].pool.block_size
        x.ok('真实KV使用非空物理页',little>0)
        oom=False
        try:Replica('B',c.root/'B',tok,models,capacity_blocks=1,load_existing=True)
        except LessonError:oom=True
        x.ok('真实小容量恢复失败',oom)
        # 单进程磁盘还在的情况下独立恢复副本；不会加载pickle代码。
        rebuild=Replica('B',c.root/'B',tok,models,load_existing=True)
        x.equal('冷恢复与WAL语义状态相等',rebuild.ledger.to_json(),c.rep['B'].ledger.to_json())
        x.ok('独立KV从token重算',not rebuild.last_recovery.get('kv_tensors_restored_from_disk',False))
        x.ok('重建后真实KV与全前向等价',rebuild.audit()['max_cache_full_logits_diff']<2e-5)
        c.promote(1,'B')
        c.rep['A']=Replica('A',c.root/'A',tok,models,load_existing=True)
        x.rejects('备机已恢复时仍必须拒绝过期epoch',lambda:c._fence(1))
        x.ok('持久控制切换后允许正确新epoch',c._fence(2) is c.rep['B'])
    # 以下场景使用真正的完整物理KV、两份文件账本，不是mock值。
    with tempfile.TemporaryDirectory() as d:
        r=Path(d)
        healthy=case_healthy(r,tok,models)
        x.ok('健康案例物理KV全部释放',healthy['all_physical_kv_released'])
        x.ok('健康案例有独立演练报告',healthy['final_control']['epoch']==1)
        dual=case_double_failure(r,tok,models)
        x.ok('双故障自动失败关门',dual['automatic_failover_blocked'])
        x.ok('安全开关阻断后续注入',dual['kill_switch_denied_next_fault'])
        x.ok('恢复必须用可信盘重建',dual['recovered_from_simulated_durable_disk'])
        shortage=case_kv_capacity(r,tok,models)
        x.ok('真实物理KV不足被验证',shortage['physical_kv_oom_verified'])
        x.ok('备用容量不足在接管前拒绝',not shortage['capacity_denial']['allowed'])
        x.ok('容量增大后才完成实际恢复',shortage['sufficient_gate']['allowed'])
        strict=case_async_loss_gate(r,tok,models)
        x.ok('未授权RPO缺口阻断切换',strict['strict_auto_denied'])
        x.equal('可信WAL补齐后零损失',strict['after_repair_loss'],0)
        manual=case_async_explicit_loss(r,tok,models)
        x.ok('人工数据丢失授权不隐藏损失',manual['loss_approval'] and manual['lost_acked_events']==1)
        x.ok('显式数据丢失必须隔离旧主',manual['old_primary_quarantined'])
        drift=case_weight_drift(r,tok,models)
        x.ok('复合模型漂移被阻断',drift['promotion_blocked_before_repair'])
        torn=case_torn_wal(r,tok,models)
        x.ok('损坏WAL不能自动切换',torn['corrupt_recovery_blocked'])
        x.equal('损坏WAL明确使用可信主盘重建',torn['repair_kind'],'explicit_replace_from_verified_primary')
        sync=case_sync_primary_loss(r,tok,models)
        x.ok('同步故障零RPO',sync['failover']['lost_acked_events']==0)
        x.ok('旧epoch围栏禁止写入',sync['old_primary_fenced'])
        x.ok('旧幂等键返回原token',sync['idempotent_retry'])
        outage=case_sync_replica_down(r,tok,models)
        x.ok('备机失联时ACK被扣留',outage['unacked_primary_write'])
        x.ok('备机恢复后同键重试不重建事件',outage['retried_original_event'])
        for name in SCENARIOS:
            x.ok('所有场景都包含完整证据起止',EvidenceTrail.verify(r/name/'chaos_evidence.jsonl')['records']>=3)
        x.equal('场景数量覆盖9类',len(SCENARIOS),9)
    return {'passed':len(x.names),'names':x.names,
            'scope':'single_process_local_lab_toy_model_real_CPU_KV; no_cloud_faults'}

In [22]:
check_report=run_checks()
print('通过的固定检查项数:',check_report['passed'])
print('实测范围:',check_report['scope'])
print('检查头五项:',check_report['names'][:5])

通过的固定检查项数: 114
实测范围: single_process_local_lab_toy_model_real_CPU_KV; no_cloud_faults
检查头五项: ['默认演练范围隔离', '默认计划合法', '默认拒绝ACK损失授权', '默认至少两页预留', '注入白名单严格限定目标']


## 12. 九案例完整离线演练

默认真正创建九套沙盒、写真实WAL、恢复KV、形成哈希证据链，再从本地磁盘加载全部九个场景并核对。所有错误必须明确失败，不能在缺证据时补写一个“通过”的模拟结论。

将下面的 `RUN_FULL_DRILLS` 改为 `False` 可只看数学和练习，但相应报告不会被伪造。

In [23]:
RUN_DIRECTORY=None
if RUN_FULL_DRILLS:
    RUN_DIRECTORY=run_experiment(OUTPUT_DIR,seed=8201)
    write_json(RUN_DIRECTORY/'checks.json',check_report)
else:
    print('已跳过九案例演练，没有伪造运行结果。')

steady_state 完成；证据 4 条


sync_primary_loss 完成；证据 7 条


sync_standby_outage 完成；证据 6 条


two_replicas_down 完成；证据 9 条


standby_kv_shortfall 完成；证据 7 条


async_rpo_gap 完成；证据 6 条


async_loss_authorized 完成；证据 6 条


model_drift_compound 完成；证据 7 条


torn_wal_compound 完成；证据 7 条


九项演练与磁盘冷恢复检查完成： lesson_82_outputs/run_20261105T074451_355127Z


## 13. 读取完成报告：区分成功恢复与安全拒绝

从真实报告中查看三类重点：

**完整恢复：** 同步故障切换零ACK损失，旧epoch被围栏，真实增量KV重算后与全前向一致。

**拒绝高风险自动切主：** 双机同时失效、RPO缺口未获授权、模型漂移、断裂WAL、灾备容量不足。

**明确的损失授权：** 异步副本落后时只有显式模拟授权才允许丢失已ACK事件，报告保留损失数和旧主隔离；不能把“接管完成”写成“数据未丢”。

In [24]:
if RUN_DIRECTORY is not None:
    results=json.loads((RUN_DIRECTORY/'scenarios.json').read_text(encoding='utf-8'))
    print('案例                       恢复后epoch    ACK水位   活跃物理KV页')
    for name,result in results.items():
        p=result['final_control']
        print(f"{name:27} {p['epoch']:6d} {p['ack_seq']:12d} {result['physical_kv_live_blocks']:12d}")
    print('明确数据损失：',results['async_loss_authorized']['lost_acked_events'])
    print('容量不足的真实KV失败：',results['standby_kv_shortfall']['physical_kv_oom_verified'])
    print('双机失效时自动接管被拒绝：',results['two_replicas_down']['automatic_failover_blocked'])
else:
    print('完整演练未运行，因此没有本次实验统计。')

案例                       恢复后epoch    ACK水位   活跃物理KV页
async_loss_authorized            2            1            2
async_rpo_gap                    2            3            3
model_drift_compound             2            1            2
standby_kv_shortfall             2            3            6
steady_state                     1           10            0
sync_primary_loss                2            5            0
sync_standby_outage              1            1            2
torn_wal_compound                2            1            2
two_replicas_down                2            1            2
明确数据损失： 1
容量不足的真实KV失败： True
双机失效时自动接管被拒绝： True


In [25]:
if RUN_DIRECTORY is not None:
    capacity=results['standby_kv_shortfall']
    print('演练容量门槛拒绝：',capacity['capacity_denial']['blockers'])
    print('需求物理块：',capacity['capacity_denial']['capacity']['total_required_blocks'])
    print('故障时可用块：',capacity['capacity_denial']['capacity']['available_blocks'])
    print('扩大页池后的恢复：',capacity['sufficient_gate']['allowed'])

演练容量门槛拒绝： ['insufficient_kv_pages']
需求物理块： 11
故障时可用块： 4
扩大页池后的恢复： True


## 14. 独立冷重建与证据篡改检测

以下单元**重新读磁盘**，不用训练／演练中驻留的模型或 KV。对每个案例重新检查：

- 报告与封存摘要一致；事件链没有缺记录或改写；
- 权重与Tokenizer/模板身份吻合；WAL和快照可信；
- 控制指针与ACK水位和所选版本正确；
- 活跃会话重放为真实KV后，增量 logits 与full forward一致；
- 页池占用和现场报告一致。

额外验证会在**临时复制品**里改动一条证据，确认不能继续验收，不修改原报告。

In [26]:
if RUN_DIRECTORY is not None:
    cold=reload_run(RUN_DIRECTORY)
    print('全部九案例从磁盘重建:',cold['all_nine_rebuilt_independently'])
    print('案例数:',len(cold['scenarios']))
    print('最大KV/full误差:',max(v['max_kv_full_logits_error'] for v in cold['scenarios'].values()))

全部九案例从磁盘重建: True
案例数: 9
最大KV/full误差: 3.5762786865234375e-07


In [27]:
if RUN_DIRECTORY is not None:
    import shutil
    original=RUN_DIRECTORY/'sync_primary_loss'/'chaos_evidence.jsonl'
    with tempfile.TemporaryDirectory() as temp:
        copy_path=Path(temp)/'trace.jsonl'
        shutil.copyfile(original,copy_path)
        original_bytes=copy_path.read_bytes()
        copy_path.write_bytes(original_bytes.replace(b'primary_down',b'primary_dowN',1))
        caught=False
        try: EvidenceTrail.verify(copy_path)
        except LessonError as exc:
            caught=True
            print('篡改副本被拒绝:',str(exc))
        assert caught
        assert original.read_bytes()!=copy_path.read_bytes()
else:
    print('本次没有运行案例；不制造伪造证据文件。')

篡改副本被拒绝: 证据内容被改写


## 15. 独立练习（默认关闭）

先手算，再补齐下面三个函数。检查器不会替你实现，开关默认 `False`。参考答案放在后面、名称不同，不覆盖你写的函数。

### 练习 A：逐会话向上取整的容量门槛

实现 `capacity_ex(active_lengths,block_size,new_lengths,reserve,available)`：活跃会话用 `(当前token数,未来最多token数)` 表示；新会话是总长度。结果必须包括 `required_blocks / headroom_blocks / allowed` 三个键，检查恰好够与差一块的边界。

In [28]:
def check_exercise_a(fn):
    # 活跃会话:(当前token数,未来最多token数)，额外新会话是总长度。
    a=fn([(5,2),(8,1)],4,[5],2,9)
    require(a=={'required_blocks':9,'headroom_blocks':0,'allowed':True},
            'A应使用每会话ceil再加reserve；不能先总长度求ceil')
    b=fn([(5,2),(8,1)],4,[5],2,8)
    require(b=={'required_blocks':9,'headroom_blocks':-1,'allowed':False},
            'A差一块必须拒绝')
    c=fn([],4,[],1,1)
    require(c['required_blocks']==1 and c['allowed'],'A空会话仍应计算安全预留')
    for bad in (lambda:fn([(1,-1)],4,[],1,9),lambda:fn([],0,[],1,9),
                lambda:fn([],4,[],True,9),lambda:fn([],4,[-1],1,9)):
        try:bad()
        except (ValueError,LessonError):pass
        else:raise LessonError('A非法预算必须报错')
    return {'exercise':'A','passed':True,'case':'per_session_ceil_plus_emergency_reserve'}


def check_exercise_b(fn):
    data=[(10,10,10,10,0,False,(0,0,True)),
          (10,8,10,8,0,False,(2,2,False)),
          (10,8,10,8,2,False,(2,2,False)),
          (10,8,10,8,2,True,(2,2,True)),
          (10,12,10,12,0,False,(0,0,True))]
    for ack,standby,tack,tstand,budget,override,expected in data:
        result=fn(ack,standby,tack,tstand,budget,override)
        require(result=={'lost_acked_events':expected[0],
                        'rpo_gap_ticks':expected[1],'allowed':expected[2]},
                'B必须同时检查丢失的ACK与RPO tick；只有显式授权才可放行实际丢失')
    for call in (lambda:fn(-1,0,0,0,0,False),lambda:fn(0,0,0,0,0,1),
                 lambda:fn(True,0,0,0,0,False)):
        try:call()
        except (LessonError,ValueError):pass
        else:raise LessonError('B错误输入未拒绝')
    return {'exercise':'B','passed':True,'case':'loss_explicitly_declared_not_silent'}


def check_exercise_c(fn):
    one=fn(ZERO_HASH,1,0,'prepare','scope_checked',{'scope':'isolated_lab_only'})
    require(set(one)==EvidenceTrail.FIELDS and one['hash']==digest({k:one[k] for k in one if k!='hash'}),
            'C第一条证据hash不正确')
    two=fn(one['hash'],2,1,'inject','primary_down',{'target':'A'})
    require(two['prev']==one['hash'] and two['seq']==2 and two['hash']!=one['hash'],
            'C第二条链必须指向前一条hash')
    changed=fn(one['hash'],2,1,'inject','primary_down',{'target':'B'})
    require(changed['hash']!=two['hash'],'C数据变化必须改变摘要')
    for test in (lambda:fn('bad',1,0,'prepare','x',{}),
                 lambda:fn(ZERO_HASH,0,0,'prepare','x',{}),
                 lambda:fn(ZERO_HASH,1,-1,'prepare','x',{})):
        try:test()
        except (LessonError,ValueError):pass
        else:raise LessonError('C非法索引/hash必须拒绝')
    return {'exercise':'C','passed':True,'case':'canonical_hash_chained_entries'}


def answer_a(active_lengths,block_size,new_lengths,reserve,available):
    require(type(active_lengths) is list and type(new_lengths) is list,'会话长度必须列表')
    require(all(type(x) is int and x>0 for x in (block_size,available)) and
            type(reserve) is int and reserve>=0,'block/容量/reserve非法')
    require(all(type(t) in (tuple,list) and len(t)==2 and
                all(type(v) is int and v>=0 for v in t) and sum(t)>0 for t in active_lengths),
            '活跃会话参数非法')
    require(all(type(n) is int and n>0 for n in new_lengths),'新会话长度非法')
    needed=sum(math.ceil((now+remaining)/block_size) for now,remaining in active_lengths)
    needed+=sum(math.ceil(n/block_size) for n in new_lengths)+reserve
    return {'required_blocks':needed,'headroom_blocks':available-needed,'allowed':available>=needed}


def answer_b(ack_seq,standby_seq,ack_tick,standby_tick,budget_ticks,manual_approved):
    require(all(type(x) is int and x>=0 for x in
                (ack_seq,standby_seq,ack_tick,standby_tick,budget_ticks)) and
            type(manual_approved) is bool,'必须使用非负整数和显式bool')
    lost=max(0,ack_seq-standby_seq)
    gap=max(0,ack_tick-standby_tick)
    return {'lost_acked_events':lost,'rpo_gap_ticks':gap,
            'allowed':(lost==0 and gap<=budget_ticks) or manual_approved}


def answer_c(previous_hash,seq,tick,phase,event,data):
    require(type(previous_hash) is str and len(previous_hash)==64 and
            all(c in '0123456789abcdef' for c in previous_hash),'前一个摘要非法')
    require(type(seq) is int and seq>0 and type(tick) is int and tick>=0,
            'seq/tick非法')
    require(phase in EvidenceTrail.PHASES and type(event) is str and bool(event) and
            type(data) is dict,'证据字段非法')
    core={'seq':seq,'tick':tick,'phase':phase,'event':event,'data':copy.deepcopy(data),
          'prev':previous_hash}
    return {**core,'hash':digest(core)}

In [29]:
def capacity_ex(active_lengths,block_size,new_lengths,reserve,available):
    raise NotImplementedError('请实现逐会话页数、紧急保留和容量门槛')

RUN_EXERCISE_A=False
if RUN_EXERCISE_A:
    print(check_exercise_a(capacity_ex))

### 练习 B：异步确认缺口不能自动洗掉

实现 `rpo_gate_ex(ack_seq,standby_seq,ack_tick,standby_tick,budget_ticks,manual_approved)`。

返回 `lost_acked_events`、`rpo_gap_ticks` 和 `allowed`。没有明确授权时，只要丢失了已ACK事件就应拒绝，即使 tick 差值落在预算内。授权需要显式bool，不可以偷用 `1`。

In [30]:
def rpo_gate_ex(ack_seq,standby_seq,ack_tick,standby_tick,budget_ticks,manual_approved):
    raise NotImplementedError('请实现RPO、ACK缺口及显式授权门槛')

RUN_EXERCISE_B=False
if RUN_EXERCISE_B:
    print(check_exercise_b(rpo_gate_ex))

### 练习 C：可审计的证据条目

实现 `trace_entry_ex(previous_hash,seq,tick,phase,event,data)`。在固定字段中加入 `prev` 和 `hash`，摘要用排序后的规范 JSON，序号和时间必须合法。修改条目内容应改变 hash，并与前一条形成链。

它只证明本地一致性，不会构造数字签名或外部不可篡改审计服务。

In [31]:
def trace_entry_ex(previous_hash,seq,tick,phase,event,data):
    raise NotImplementedError('请实现带前驱hash的审计条目')

RUN_EXERCISE_C=False
if RUN_EXERCISE_C:
    print(check_exercise_c(trace_entry_ex))

## 16. 独立参考答案检查

上一节已经定义了 `answer_a / answer_b / answer_c`，但没有修改你的练习函数。建议先自己写完，再看参考实现和输出。

In [32]:
print('练习A:',check_exercise_a(answer_a))
print('练习B:',check_exercise_b(answer_b))
print('练习C:',check_exercise_c(answer_c))

练习A: {'exercise': 'A', 'passed': True, 'case': 'per_session_ceil_plus_emergency_reserve'}
练习B: {'exercise': 'B', 'passed': True, 'case': 'loss_explicitly_declared_not_silent'}
练习C: {'exercise': 'C', 'passed': True, 'case': 'canonical_hash_chained_entries'}


## 17. 命令行、可复现性与本课验收

完整 `.py` 和本 Notebook 使用同一套定义，无需彼此导入。

```bash
python lesson_82_chaos_drills_dr_capacity_evidence.py --checks-only
python lesson_82_chaos_drills_dr_capacity_evidence.py --math-only
python lesson_82_chaos_drills_dr_capacity_evidence.py
python lesson_82_chaos_drills_dr_capacity_evidence.py --reload-run lesson_82_outputs/实际运行目录
```

**验收问题**：

1. 为什么备用容量应按每个请求的未来最长 KV 长度估算，并保留余量？
2. 为什么 sync 下“主盘已fsync，但备机未fsync”仍不能给出同步ACK？
3. 为什么双机同时 down 时，安全停止比假装切主成功更正确？
4. 为什么异步策略即使获得人工数据损失批准，也必须记录已ACK数据损失与客户端对账？
5. 为什么原模型权重或 chat template 改变后，不能把已有 token IDs 当作可复用的同一 KV？
6. 为什么本地hash链不等于可信外部签名？
7. 为什么本课逻辑 RTO 不应直接报告为 GPU 毫秒或者生产 RTO 承诺？

本课默认使用随机微型模型、单进程两个目录、CPU Tensor 物理KV；没有网络分区、真实主机、分布式共识、真实IAM、分布式对象存储、GPU显存或在线生产流量。

**下一课：第 83 课——推理服务变更管理、配置漂移与端到端恢复验收。** 把运行时模型/路由配置、版本发布和灾备演练证据组成更严格的发布门槛。

## 主要参考资料

- [AWS Fault Injection Service：实验范围、故障动作、安全控制](https://docs.aws.amazon.com/fis/latest/userguide/what-is.html)
- [AWS FIS：停止条件](https://docs.aws.amazon.com/fis/latest/userguide/stop-conditions.html)
- [Microsoft Azure Well-Architected：可靠性测试、SLO/RPO/RTO 与范围](https://learn.microsoft.com/en-us/azure/well-architected/reliability/reliability-test)
- [AWS Well-Architected：常规恢复测试与RPO/RTO](https://docs.aws.amazon.com/wellarchitected/2024-06-27/framework/rel-failmgmt.html)
- [PyTorch：复现性限制](https://docs.pytorch.org/docs/stable/notes/randomness.html)

资料用于原理对照；本课数值来自本地真实执行，不代表这些服务商的实际运行数据。

In [33]:
def main(argv=None):
    parser=argparse.ArgumentParser(description=__doc__)
    group=parser.add_mutually_exclusive_group()
    group.add_argument('--checks-only',action='store_true')
    group.add_argument('--math-only',action='store_true')
    group.add_argument('--reload-run',type=Path)
    parser.add_argument('--output-dir',type=Path,default=Path('lesson_82_outputs'))
    parser.add_argument('--seed',type=int,default=8201)
    args=parser.parse_args(argv)
    torch.set_num_threads(1)
    if args.reload_run:
        print(json.dumps(reload_run(args.reload_run),ensure_ascii=False,indent=2))
        return 0
    if args.math_only:
        print(json.dumps({'capacity_example':answer_a([(5,2),(8,1)],4,[5],2,9),
              'rpo_example':answer_b(10,8,10,8,2,False),
              'scenario_names':list(SCENARIOS),
              'physical_kv_bytes_formula':'layers * 2(K,V) * heads * head_dim * dtype_bytes * block_size',
              'RTO_toy_tick_formula':'2 + ceil(verified_WAL / records_per_tick) + ceil(rebuilt_tokens / tokens_per_tick)',
              'not_real_hosts_or_seconds':True},ensure_ascii=False,indent=2))
        return 0
    checks=run_checks()
    print(f"固定检查通过：{checks['passed']}项。真实CPU KV、隔离双副本，不是云端故障注入。")
    if args.checks_only:return 0
    folder=run_experiment(args.output_dir,args.seed)
    write_json(folder/'checks.json',checks)
    print('独立重载9个场景，完成。')
    return 0